# 数据准备流水线（Steps 1–13）

本 notebook **整段嵌入** `ELSE/scripts/` 下已重命名的脚本，与终端 `python ELSE/scripts/<文件>.py` 行为一致（需 `config.py`）。

**运行方式：** 在**仓库根目录**打开本 notebook，须**首先**运行下方 **「0. 环境与路径」** 中的代码单元（注入 `sys.path`，否则 `from config import …` 失败），再按需顺序执行各 Step 代码单元。各脚本末尾的 `if __name__ == "__main__": main()` 在 notebook 中需**手动调用** `main()` 或在终端运行对应 `.py` 文件。

| Step | 脚本文件 | 说明 |
|------|----------|------|
| Step 1/13 — GitHub 开放 AI 仓库发现 | `step_01_github_discovery.py` | 原 `step1a_filter_github.py` |
| Step 2/13 — Hugging Face 项目发现 | `step_02_huggingface_discovery.py` | 原 `step1b_filter_huggingface.py` |
| Step 3/13 — 合并 GitHub + HF 候选 | `step_03_merge_candidates.py` | 原 `step1c_merge_candidates.py` |
| Step 4/13 — GitHub owner 地理位置 | `step_04_github_locations.py` | 原 `step3a_fetch_github_locations.py` |
| Step 5/13 — HF 作者地理位置 | `step_05_hf_author_locations.py` | 原 `step3a_hf_fetch_user_locations.py` |
| Step 6/13 — 清洗并映射城市 | `step_06_clean_map_locations.py` | 原 `step3b_clean_and_map_locations.py` |
| Step 7/13 — Nominatim 补地理编码 | `step_07_geocode_unmatched.py` | 原 `step3c_geocode_unmatched.py` |
| Step 8/13 — 构建城市列表 | `step_08_build_city_list.py` | 原 `step3d_build_city_list.py` |
| Step 9/13 — 贡献者与参与事件 | `step_09_contributors_participation.py` | 原 `step4_fetch_contributors.py` |
| Step 10/13 — HF base_model 衍生边 | `step_10_hf_derivation_edges.py` | 原 `step5b_build_hf_derivation_edges.py` · 须在 Step 11 前运行 |
| Step 11/13 — 三张核心分析表 | `step_11_core_tables.py` | 原 `step5_build_core_tables.py` |
| Step 12/13 — 外部社会经济属性 | `step_12_augment_external.py` | 原 `step6_augment_city_attributes.py` |
| Step 13/13 — 建模用衍生特征 | `step_13_enrich_attributes.py` | 原 `step6b_enrich_city_features.py` |

**依赖：** `ELSE/scripts/config.py`（未嵌入；通过 `sys.path` 加载）。

**离线链（示例）：** 在已有原始 CSV 的前提下，可依次调用 Step 3→6→8→10→11→12→13 的 `main()`。

---
## 0. 环境与路径

以下说明与**紧随其后的代码单元**一致；请先阅读，再**运行该代码单元**（不要跳过）。

- **工作目录**：Jupyter 的当前目录应为仓库根目录（与 `ELSE/`、`data/` 同级）。
- **`_ROOT`**：`Path.cwd()`，即项目根。
- **`_SCRIPTS`**：`ELSE/scripts`，内含 `config.py` 与各 `step_*.py`。
- **`sys.path`**：将 `_SCRIPTS` 置于首位后，`import config` 时 `config.PROJECT_ROOT` 仍为「`ELSE` 的上一级」，与终端执行脚本行为一致。

下面代码块内容与下一单元**相同**（便于复制到其它环境或核对）：

```python
from pathlib import Path
import sys

_ROOT = Path.cwd().resolve()
_SCRIPTS = _ROOT / "ELSE" / "scripts"
_CFG = _SCRIPTS / "config.py"
if not _CFG.exists():
    raise RuntimeError(f"请在仓库根目录打开 notebook：找不到 {_CFG}")

_SCRIPTS_STR = str(_SCRIPTS.resolve())
if _SCRIPTS_STR not in sys.path:
    sys.path.insert(0, _SCRIPTS_STR)

print(f"OK: sys.path 已包含 ELSE/scripts，config 自 {_CFG}")
```


In [ ]:
# 0. 环境与路径 — 可运行单元（须先执行）
from pathlib import Path
import sys

_ROOT = Path.cwd().resolve()
_SCRIPTS = _ROOT / "ELSE" / "scripts"
_CFG = _SCRIPTS / "config.py"
if not _CFG.exists():
    raise RuntimeError(f"请在仓库根目录打开 notebook：找不到 {_CFG}")

_SCRIPTS_STR = str(_SCRIPTS.resolve())
if _SCRIPTS_STR not in sys.path:
    sys.path.insert(0, _SCRIPTS_STR)

print(f"OK: sys.path 已包含 ELSE/scripts，config 自 {_CFG}")


---
## Step 1/13 — GitHub 开放 AI 仓库发现

**原 `step1a_filter_github.py`**

源文件：`ELSE/scripts/step_01_github_discovery.py`

执行：`import step_01_github_discovery as _m; _m.main()` 或在终端 `python ELSE/scripts/step_01_github_discovery.py`。


In [ ]:
"""
Step 1a – Discover and filter open-AI-related GitHub repositories.

Strategy
--------
1. Use the GitHub Search API to find repos matching AI-related keywords,
   created within the study window (2022-01 to 2025-12).
2. For each candidate repo, collect metadata (stars, forks, topics,
   description, owner location, created_at, etc.).
3. Apply the open-AI relevance rule (keyword match on name + description +
   topics) and record match evidence & confidence.
4. Apply the prominence threshold (stars / forks).
5. Save results to  data/raw/github/github_candidates.csv

Rate-limit handling
-------------------
- The Search API allows 30 requests/min (authenticated) with up to 1000
  results per query.  We split queries by creation-date month to stay under
  the 1000-result cap, and sleep between pages to respect rate limits.
"""

import time
import csv
import re
import sys
import requests
from datetime import date, timedelta
from pathlib import Path

from config import (
    GITHUB_TOKEN,
    DATA_RAW,
    TIME_START,
    TIME_END,
    AI_INCLUDE_KEYWORDS,
    AI_WEAK_KEYWORDS,
    AI_EXCLUDE_KEYWORDS,
    GITHUB_PROMINENCE,
)

HEADERS = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

SEARCH_URL = "https://api.github.com/search/repositories"
PER_PAGE = 100
MAX_PAGES = 10  # 1000 results per query slice

# We search with a curated set of high-signal queries (each is a GitHub search
# string).  Using too many keywords at once hits the query-length limit, so we
# group them into batches.
SEARCH_QUERIES = [
    "llm OR large-language-model OR language-model",
    "transformer OR attention-mechanism",
    "gpt OR chatgpt OR instruction-tuning",
    "diffusion OR stable-diffusion OR latent-diffusion",
    "text-generation OR code-generation OR image-generation",
    "generative-ai OR text-to-image OR text-to-video",
    "multimodal OR vision-language OR clip",
    "ai-agent OR autonomous-agent OR rag OR retrieval-augmented-generation",
    "embedding OR sentence-embedding OR vector-database OR semantic-search",
    "natural-language-processing OR nlp OR question-answering OR summarization",
    "computer-vision OR object-detection OR image-segmentation",
    "speech-recognition OR automatic-speech-recognition OR text-to-speech",
    "deep-learning OR neural-network OR reinforcement-learning",
    "open-weight OR llama OR mistral OR falcon OR qwen OR deepseek",
    "quantization OR gguf OR vllm OR model-serving OR model-inference",
    "whisper OR segment-anything OR stable-audio",
    "fine-tuning OR rlhf OR prompt-engineering",
    "mlops OR onnx OR tensorrt",
]


def _generate_half_year_windows(start: str, end: str):
    """Yield (start_date, end_date) for each ~6-month window within [start, end]."""
    sd = date.fromisoformat(start)
    ed = date.fromisoformat(end)
    cur = sd
    while cur <= ed:
        half_end = date(cur.year, 6, 30) if cur.month <= 6 else date(cur.year, 12, 31)
        win_end = min(half_end, ed)
        yield cur.isoformat(), win_end.isoformat()
        nxt = win_end + timedelta(days=1)
        if nxt > ed:
            break
        cur = nxt


def _wait_for_rate_limit(response: requests.Response):
    """Sleep until the rate-limit window resets if we've run out of quota."""
    remaining = int(response.headers.get("X-RateLimit-Remaining", 1))
    if remaining == 0:
        reset_ts = int(response.headers.get("X-RateLimit-Reset", 0))
        sleep_sec = max(reset_ts - int(time.time()), 1) + 2
        print(f"  ⏳ Rate-limited. Sleeping {sleep_sec}s …")
        time.sleep(sleep_sec)


def _match_ai_relevance(name, desc, topics):
    """
    Check whether a repo is open-AI-related.
    Returns (is_relevant: bool, evidence: str, confidence: str).
    """
    text = f"{name} {desc} {' '.join(topics)}".lower()
    # normalise separators so "text_generation" matches "text-generation"
    text_norm = re.sub(r"[_/.]", "-", text)

    matched = []
    for kw in AI_INCLUDE_KEYWORDS:
        kw_lower = kw.lower()
        if kw_lower in text_norm:
            matched.append(kw_lower)

    if not matched:
        return False, "", "none"

    # Exclude false positives
    for ex in AI_EXCLUDE_KEYWORDS:
        if ex.lower() in text_norm:
            return False, f"excluded:{ex}", "none"

    # If only weak keywords matched, require a second confirming signal
    strong = [m for m in matched if m not in AI_WEAK_KEYWORDS]
    if not strong and len(matched) < 2:
        # Allow "agent" if description has AI context words
        ai_context = re.search(
            r"(?i)\b(ai|artificial.intelligence|llm|language.model|gpt|autonom|coding|code.gen|reasoning)\b",
            text_norm,
        )
        if ai_context:
            matched.append(f"context:{ai_context.group()}")
        else:
            return False, f"weak-only:{matched}", "low"

    confidence = "high" if len(matched) >= 2 or strong else "medium"
    evidence = "; ".join(sorted(set(matched)))
    return True, evidence, confidence


def _is_prominent(stars: int, forks: int) -> bool:
    s = stars >= GITHUB_PROMINENCE["stars_min"]
    f = forks >= GITHUB_PROMINENCE["forks_min"]
    if GITHUB_PROMINENCE["logic"] == "or":
        return s or f
    return s and f


def search_github():
    """Run all search queries and collect unique repos."""
    seen_ids = set()
    rows = []

    total_queries = len(SEARCH_QUERIES)

    for qi, query in enumerate(SEARCH_QUERIES, 1):
        windows = list(_generate_half_year_windows(TIME_START, TIME_END))
        for wi, (ws, we) in enumerate(windows):
            q = f"{query} created:{ws}..{we}"
            print(f"[{qi}/{total_queries}] window {wi+1}/{len(windows)}: {q[:80]}…")

            for page in range(1, MAX_PAGES + 1):
                params = {
                    "q": q,
                    "sort": "stars",
                    "order": "desc",
                    "per_page": PER_PAGE,
                    "page": page,
                }
                resp = requests.get(SEARCH_URL, headers=HEADERS, params=params)
                _wait_for_rate_limit(resp)

                if resp.status_code == 403:
                    print("  ⚠️  403 – sleeping 60s")
                    time.sleep(60)
                    resp = requests.get(SEARCH_URL, headers=HEADERS, params=params)

                if resp.status_code != 200:
                    print(f"  ⚠️  HTTP {resp.status_code}, skipping page")
                    break

                data = resp.json()
                items = data.get("items", [])
                if not items:
                    break

                for repo in items:
                    rid = repo["id"]
                    if rid in seen_ids:
                        continue
                    seen_ids.add(rid)

                    name = repo.get("name", "")
                    desc = repo.get("description") or ""
                    topics = repo.get("topics", [])
                    stars = repo.get("stargazers_count", 0)
                    forks = repo.get("forks_count", 0)

                    is_ai, evidence, confidence = _match_ai_relevance(
                        name, desc, topics
                    )

                    rows.append(
                        {
                            "project_id": f"gh_{rid}",
                            "platform": "GitHub",
                            "repo_full_name": repo.get("full_name", ""),
                            "project_name": name,
                            "description": desc[:500],
                            "topics": "|".join(topics),
                            "language": repo.get("language") or "",
                            "stars": stars,
                            "forks": forks,
                            "watchers": repo.get("watchers_count", 0),
                            "open_issues": repo.get("open_issues_count", 0),
                            "created_at": repo.get("created_at", ""),
                            "updated_at": repo.get("updated_at", ""),
                            "pushed_at": repo.get("pushed_at", ""),
                            "owner_login": repo.get("owner", {}).get("login", ""),
                            "owner_type": repo.get("owner", {}).get("type", ""),
                            "license": (repo.get("license") or {}).get(
                                "spdx_id", ""
                            ),
                            "homepage": repo.get("homepage") or "",
                            "html_url": repo.get("html_url", ""),
                            "open_ai_related": int(is_ai),
                            "ai_evidence": evidence,
                            "ai_confidence": confidence,
                            "prominent_flag": int(
                                is_ai and _is_prominent(stars, forks)
                            ),
                        }
                    )

                time.sleep(2.5)  # respect rate limit

    return rows


def save_csv(rows, path):
    if not rows:
        print("⚠️  No rows to save.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ Saved {len(rows)} rows → {path}")


def reevaluate():
    """Re-evaluate AI relevance on an existing github_candidates.csv
    using the current keyword rules. No API calls needed."""
    import pandas as pd

    print("=" * 60)
    print("Step 1a [re-evaluate]: update flags with current keywords")
    print("=" * 60)

    gh_path = DATA_RAW / "github" / "github_candidates.csv"
    df = pd.read_csv(gh_path, dtype=str).fillna("")

    old_ai = (df["open_ai_related"] == "1").sum()
    old_prom = (df["prominent_flag"] == "1").sum()

    for idx, row in df.iterrows():
        name = row["project_name"]
        desc = row["description"]
        topics = row["topics"].split("|") if row["topics"] else []
        stars = int(row["stars"]) if row["stars"] else 0
        forks = int(row["forks"]) if row["forks"] else 0

        is_ai, evidence, confidence = _match_ai_relevance(name, desc, topics)
        df.at[idx, "open_ai_related"] = str(int(is_ai))
        df.at[idx, "ai_evidence"] = evidence
        df.at[idx, "ai_confidence"] = confidence
        df.at[idx, "prominent_flag"] = str(int(is_ai and _is_prominent(stars, forks)))

    df.to_csv(gh_path, index=False)

    new_ai = (df["open_ai_related"] == "1").sum()
    new_prom = (df["prominent_flag"] == "1").sum()
    print(f"\n  open_ai_related: {old_ai} → {new_ai} ({new_ai - old_ai:+d})")
    print(f"  prominent_flag:  {old_prom} → {new_prom} ({new_prom - old_prom:+d})")


def main():
    print("=" * 60)
    print("Step 1a: GitHub open-AI project discovery")
    print("=" * 60)

    if "--reevaluate" in sys.argv:
        reevaluate()
        return

    if not GITHUB_TOKEN:
        print(
            "⚠️  GITHUB_TOKEN not set. Requests will be severely rate-limited.\n"
            "   export GITHUB_TOKEN='ghp_…' before running."
        )

    rows = search_github()

    out_path = DATA_RAW / "github" / "github_candidates.csv"
    save_csv(rows, out_path)

    # Summary
    total = len(rows)
    ai_yes = sum(1 for r in rows if r["open_ai_related"])
    prominent = sum(1 for r in rows if r["prominent_flag"])
    print(f"\n📊 Summary: {total} repos collected")
    print(f"   open_ai_related = 1 : {ai_yes}")
    print(f"   prominent_flag  = 1 : {prominent}")


if __name__ == "__main__":
    main()


---
## Step 2/13 — Hugging Face 项目发现

**原 `step1b_filter_huggingface.py`**

源文件：`ELSE/scripts/step_02_huggingface_discovery.py`

执行：`import step_02_huggingface_discovery as _m; _m.main()` 或在终端 `python ELSE/scripts/step_02_huggingface_discovery.py`。


In [ ]:
"""
Step 1b – Discover and filter open-AI-related projects on Hugging Face.

Strategy
--------
1. Use the Hugging Face Hub API (`huggingface_hub` library) to list:
   - Models
   - Datasets
   - Spaces
   that were created within the study window (2022-01 to 2025-12).
2. For each object, collect metadata (downloads, likes, pipeline_tag,
   tags, author, etc.).
3. Apply open-AI relevance rules (pipeline_tag match + tag/keyword match)
   and record evidence & confidence.
4. Apply prominence threshold (downloads / likes).
5. Save results to  data/raw/huggingface/hf_candidates.csv
"""

import csv
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Optional, List

from config import (
    HF_TOKEN,
    DATA_RAW,
    TIME_START,
    TIME_END,
    AI_INCLUDE_KEYWORDS,
    AI_WEAK_KEYWORDS,
    AI_EXCLUDE_KEYWORDS,
    HF_AI_PIPELINE_TAGS,
    HF_PROMINENCE,
)

try:
    from huggingface_hub import HfApi
except ImportError:
    print("❌ Please install huggingface_hub:  pip install huggingface_hub")
    sys.exit(1)

api = HfApi(token=HF_TOKEN or None)

START_DT = datetime.strptime(TIME_START, "%Y-%m-%d")
END_DT = datetime.strptime(TIME_END, "%Y-%m-%d")


def _parse_created_at(obj) -> Optional[datetime]:
    """Extract and parse created_at from a HF Hub object."""
    created = getattr(obj, "created_at", None) or getattr(obj, "lastModified", None)
    if isinstance(created, str):
        try:
            created = datetime.fromisoformat(created.replace("Z", "+00:00")).replace(tzinfo=None)
        except ValueError:
            return None
    if isinstance(created, datetime) and created.tzinfo is not None:
        created = created.replace(tzinfo=None)
    return created


def _in_window(created: Optional[datetime]) -> bool:
    if created is None:
        return False
    if created.tzinfo is not None:
        created = created.replace(tzinfo=None)
    return START_DT <= created <= END_DT


def _match_ai_relevance(
    model_id: str, tags: List[str], pipeline_tag: Optional[str], card_data_tags: List[str]
):
    """
    Returns (is_relevant, evidence, confidence).
    """
    all_tags = [t.lower() for t in tags + card_data_tags]
    text = f"{model_id} {' '.join(all_tags)}".lower()
    text_norm = re.sub(r"[_/.]", "-", text)

    matched = []

    # pipeline_tag is a strong signal on HF
    if pipeline_tag and pipeline_tag.lower() in [t.lower() for t in HF_AI_PIPELINE_TAGS]:
        matched.append(f"pipeline:{pipeline_tag}")

    for kw in AI_INCLUDE_KEYWORDS:
        if kw.lower() in text_norm:
            matched.append(kw.lower())

    if not matched:
        return False, "", "none"

    for ex in AI_EXCLUDE_KEYWORDS:
        if ex.lower() in text_norm:
            return False, f"excluded:{ex}", "none"

    strong = [m for m in matched if m not in AI_WEAK_KEYWORDS and not m.startswith("pipeline:")]
    pipeline_match = any(m.startswith("pipeline:") for m in matched)

    if pipeline_match:
        confidence = "high"
    elif strong:
        confidence = "high" if len(strong) >= 2 else "medium"
    else:
        confidence = "low"

    evidence = "; ".join(sorted(set(matched)))
    return True, evidence, confidence


def _is_prominent(downloads: int, likes: int) -> bool:
    d = downloads >= HF_PROMINENCE["downloads_min"]
    li = likes >= HF_PROMINENCE["likes_min"]
    if HF_PROMINENCE["logic"] == "or":
        return d or li
    return d and li


def _safe_str(val) -> str:
    if val is None:
        return ""
    return str(val)


def collect_models() -> list[dict]:
    """Iterate over HF models and collect candidates."""
    print("📦 Fetching models …")
    rows = []
    count = 0
    for model in api.list_models(
        sort="downloads",
        limit=None,
        full=True,
        cardData=True,
    ):
        created = _parse_created_at(model)

        if not _in_window(created):
            # Models are sorted by downloads desc; once we've passed the window
            # we still continue because creation date isn't monotonic with downloads.
            count += 1
            if count > 200000:
                break
            continue

        model_id = getattr(model, "modelId", "") or getattr(model, "id", "")
        tags = list(getattr(model, "tags", []) or [])
        pipeline_tag = _safe_str(getattr(model, "pipeline_tag", None))
        card_tags = list(getattr(model, "cardData", {}).get("tags", []) if getattr(model, "cardData", None) else [])
        downloads = getattr(model, "downloads", 0) or 0
        likes = getattr(model, "likes", 0) or 0
        author = model_id.split("/")[0] if "/" in model_id else ""

        is_ai, evidence, confidence = _match_ai_relevance(
            model_id, tags, pipeline_tag, card_tags
        )

        rows.append(
            {
                "project_id": f"hf_model_{model_id.replace('/', '__')}",
                "platform": "HuggingFace",
                "hf_type": "model",
                "hf_id": model_id,
                "project_name": model_id.split("/")[-1] if "/" in model_id else model_id,
                "author": author,
                "pipeline_tag": pipeline_tag,
                "tags": "|".join(tags[:30]),
                "downloads": downloads,
                "likes": likes,
                "created_at": _safe_str(created),
                "open_ai_related": int(is_ai),
                "ai_evidence": evidence,
                "ai_confidence": confidence,
                "prominent_flag": int(is_ai and _is_prominent(downloads, likes)),
            }
        )
        count += 1
        if count % 5000 == 0:
            print(f"  … scanned {count} models, kept {len(rows)} candidates")

    print(f"  ✔ Models done: {len(rows)} candidates from {count} scanned")
    return rows


def collect_datasets() -> list[dict]:
    """Iterate over HF datasets and collect candidates."""
    print("📦 Fetching datasets …")
    rows = []
    count = 0
    for ds in api.list_datasets(
        sort="downloads",
        limit=None,
        full=True,
    ):
        created = _parse_created_at(ds)

        if not _in_window(created):
            count += 1
            if count > 200000:
                break
            continue

        ds_id = getattr(ds, "id", "")
        tags = list(getattr(ds, "tags", []) or [])
        downloads = getattr(ds, "downloads", 0) or 0
        likes = getattr(ds, "likes", 0) or 0
        author = ds_id.split("/")[0] if "/" in ds_id else ""

        is_ai, evidence, confidence = _match_ai_relevance(ds_id, tags, None, [])

        rows.append(
            {
                "project_id": f"hf_dataset_{ds_id.replace('/', '__')}",
                "platform": "HuggingFace",
                "hf_type": "dataset",
                "hf_id": ds_id,
                "project_name": ds_id.split("/")[-1] if "/" in ds_id else ds_id,
                "author": author,
                "pipeline_tag": "",
                "tags": "|".join(tags[:30]),
                "downloads": downloads,
                "likes": likes,
                "created_at": _safe_str(created),
                "open_ai_related": int(is_ai),
                "ai_evidence": evidence,
                "ai_confidence": confidence,
                "prominent_flag": int(is_ai and _is_prominent(downloads, likes)),
            }
        )
        count += 1
        if count % 5000 == 0:
            print(f"  … scanned {count} datasets, kept {len(rows)} candidates")

    print(f"  ✔ Datasets done: {len(rows)} candidates from {count} scanned")
    return rows


def collect_spaces() -> list[dict]:
    """Iterate over HF Spaces and collect candidates."""
    print("📦 Fetching Spaces …")
    rows = []
    count = 0
    for sp in api.list_spaces(
        sort="likes",
        limit=None,
        full=True,
    ):
        created = _parse_created_at(sp)

        if not _in_window(created):
            count += 1
            if count > 100000:
                break
            continue

        sp_id = getattr(sp, "id", "")
        tags = list(getattr(sp, "tags", []) or [])
        likes = getattr(sp, "likes", 0) or 0
        author = sp_id.split("/")[0] if "/" in sp_id else ""

        is_ai, evidence, confidence = _match_ai_relevance(sp_id, tags, None, [])

        rows.append(
            {
                "project_id": f"hf_space_{sp_id.replace('/', '__')}",
                "platform": "HuggingFace",
                "hf_type": "space",
                "hf_id": sp_id,
                "project_name": sp_id.split("/")[-1] if "/" in sp_id else sp_id,
                "author": author,
                "pipeline_tag": "",
                "tags": "|".join(tags[:30]),
                "downloads": 0,
                "likes": likes,
                "created_at": _safe_str(created),
                "open_ai_related": int(is_ai),
                "ai_evidence": evidence,
                "ai_confidence": confidence,
                "prominent_flag": int(
                    is_ai and likes >= HF_PROMINENCE["likes_min"]
                ),
            }
        )
        count += 1
        if count % 5000 == 0:
            print(f"  … scanned {count} spaces, kept {len(rows)} candidates")

    print(f"  ✔ Spaces done: {len(rows)} candidates from {count} scanned")
    return rows


def save_csv(rows: list[dict], path: Path):
    if not rows:
        print("⚠️  No rows to save.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ Saved {len(rows)} rows → {path}")


def main():
    print("=" * 60)
    print("Step 1b: Hugging Face open-AI project discovery")
    print("=" * 60)

    all_rows = []
    all_rows.extend(collect_models())
    all_rows.extend(collect_datasets())
    all_rows.extend(collect_spaces())

    out_path = DATA_RAW / "huggingface" / "hf_candidates.csv"
    save_csv(all_rows, out_path)

    total = len(all_rows)
    ai_yes = sum(1 for r in all_rows if r["open_ai_related"])
    prominent = sum(1 for r in all_rows if r["prominent_flag"])
    print(f"\n📊 Summary: {total} HF objects collected")
    print(f"   open_ai_related = 1 : {ai_yes}")
    print(f"   prominent_flag  = 1 : {prominent}")


if __name__ == "__main__":
    main()


---
## Step 3/13 — 合并 GitHub + HF 候选

**原 `step1c_merge_candidates.py`**

源文件：`ELSE/scripts/step_03_merge_candidates.py`

执行：`import step_03_merge_candidates as _m; _m.main()` 或在终端 `python ELSE/scripts/step_03_merge_candidates.py`。


In [ ]:
"""
Step 1c – Merge GitHub and Hugging Face candidates into a unified
           project filtering result table.

Outputs
-------
data/processed/project_filtering_result.csv      – all candidates with flags
data/processed/prominent_projects_master.csv      – only prominent_flag == 1
"""

import sys
from pathlib import Path

import pandas as pd

from config import DATA_RAW, DATA_PROCESSED

GH_PATH = DATA_RAW / "github" / "github_candidates.csv"
HF_PATH = DATA_RAW / "huggingface" / "hf_candidates.csv"


def _align_to_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing columns and reorder to the unified schema."""
    cols = _unified_cols()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df[cols]


def _load_github(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    df = df.rename(
        columns={
            "repo_full_name": "full_id",
            "stars": "metric_stars",
            "forks": "metric_forks",
        }
    )
    df["metric_downloads"] = ""
    df["metric_likes"] = ""
    df["hf_type"] = ""
    return _align_to_schema(df)


def _load_hf(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    df = df.rename(
        columns={
            "hf_id": "full_id",
            "downloads": "metric_downloads",
            "likes": "metric_likes",
        }
    )
    df["metric_stars"] = ""
    df["metric_forks"] = ""
    return _align_to_schema(df)


def _unified_cols() -> list[str]:
    return [
        "project_id",
        "platform",
        "full_id",
        "project_name",
        "hf_type",
        "tags",
        "metric_stars",
        "metric_forks",
        "metric_downloads",
        "metric_likes",
        "created_at",
        "open_ai_related",
        "ai_evidence",
        "ai_confidence",
        "prominent_flag",
    ]


def main():
    print("=" * 60)
    print("Step 1c: Merge GitHub + HuggingFace candidates")
    print("=" * 60)

    frames = []
    if GH_PATH.exists():
        gh = _load_github(GH_PATH)
        print(f"  GitHub candidates : {len(gh)}")
        frames.append(gh)
    else:
        print(f"  ⚠️  {GH_PATH} not found – skipping GitHub")

    if HF_PATH.exists():
        hf = _load_hf(HF_PATH)
        print(f"  HF candidates     : {len(hf)}")
        frames.append(hf)
    else:
        print(f"  ⚠️  {HF_PATH} not found – skipping HuggingFace")

    if not frames:
        print("❌ No candidate files found. Run step1a / step1b first.")
        return

    merged = pd.concat(frames, ignore_index=True)

    # Ensure flag columns are numeric
    for col in ["open_ai_related", "prominent_flag"]:
        merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(0).astype(int)

    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

    # Full result
    full_path = DATA_PROCESSED / "project_filtering_result.csv"
    merged.to_csv(full_path, index=False)
    print(f"\n✅ Full result  : {len(merged)} rows → {full_path}")

    # Prominent-only subset
    prominent = merged[merged["prominent_flag"] == 1].copy()
    prom_path = DATA_PROCESSED / "prominent_projects_master.csv"
    prominent.to_csv(prom_path, index=False)
    print(f"✅ Prominent only: {len(prominent)} rows → {prom_path}")

    # Summary
    print("\n📊 Breakdown:")
    print(merged.groupby(["platform", "open_ai_related", "prominent_flag"]).size()
          .unstack(fill_value=0).to_string())

    ai_conf = merged[merged["open_ai_related"] == 1]["ai_confidence"].value_counts()
    print(f"\n📊 AI confidence distribution (open_ai_related=1):\n{ai_conf.to_string()}")


if __name__ == "__main__":
    main()


---
## Step 4/13 — GitHub owner 地理位置

**原 `step3a_fetch_github_locations.py`**

源文件：`ELSE/scripts/step_04_github_locations.py`

执行：`import step_04_github_locations as _m; _m.main()` 或在终端 `python ELSE/scripts/step_04_github_locations.py`。


In [ ]:
"""
Step 3a – Fetch location info for GitHub users/orgs that own prominent repos.

For each unique owner_login in the prominent GitHub projects, call the
GitHub Users API to retrieve the `location` field.

Output: data/raw/github/github_owner_locations.csv
"""

import csv
import sys
import time
from pathlib import Path

import pandas as pd
import requests

from config import GITHUB_TOKEN, DATA_RAW, DATA_PROCESSED

HEADERS = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"


def _wait_for_rate_limit(resp):
    remaining = int(resp.headers.get("X-RateLimit-Remaining", 1))
    if remaining < 5:
        reset_ts = int(resp.headers.get("X-RateLimit-Reset", 0))
        sleep_sec = max(reset_ts - int(time.time()), 1) + 2
        print(f"  ⏳ Rate-limited ({remaining} left). Sleeping {sleep_sec}s …")
        time.sleep(sleep_sec)


def fetch_user_location(login, owner_type):
    """Fetch location from /users/{login} or /orgs/{login}."""
    if owner_type == "Organization":
        url = f"https://api.github.com/orgs/{login}"
    else:
        url = f"https://api.github.com/users/{login}"

    resp = requests.get(url, headers=HEADERS)
    _wait_for_rate_limit(resp)

    if resp.status_code == 404:
        return {"login": login, "location": "", "name": "", "company": "",
                "bio": "", "type": owner_type, "status": "not_found"}
    if resp.status_code != 200:
        time.sleep(5)
        return {"login": login, "location": "", "name": "", "company": "",
                "bio": "", "type": owner_type, "status": f"http_{resp.status_code}"}

    data = resp.json()
    return {
        "login": login,
        "location": data.get("location") or "",
        "name": data.get("name") or "",
        "company": data.get("company") or "",
        "bio": (data.get("bio") or "")[:200],
        "type": data.get("type") or owner_type,
        "status": "ok",
    }


def main():
    print("=" * 60)
    print("Step 3a: Fetch GitHub owner locations")
    print("=" * 60)

    if not GITHUB_TOKEN:
        print("⚠️  GITHUB_TOKEN not set. Will be heavily rate-limited.")

    gh_csv = DATA_RAW / "github" / "github_candidates.csv"
    df = pd.read_csv(gh_csv, dtype=str)
    prominent = df[df["prominent_flag"] == "1"]
    owners = prominent[["owner_login", "owner_type"]].drop_duplicates()
    print(f"  Prominent repos: {len(prominent)}")
    print(f"  Unique owners:   {len(owners)}")

    # Resume support: skip already-fetched logins
    out_path = DATA_RAW / "github" / "github_owner_locations.csv"
    done_logins = set()
    existing_rows = []
    if out_path.exists():
        existing = pd.read_csv(out_path, dtype=str)
        done_logins = set(existing["login"].tolist())
        existing_rows = existing.to_dict("records")
        print(f"  Already fetched:  {len(done_logins)} (will resume)")

    rows = list(existing_rows)
    todo = [(r["owner_login"], r["owner_type"])
            for _, r in owners.iterrows() if r["owner_login"] not in done_logins]
    total = len(todo)
    print(f"  Remaining to fetch: {total}\n")

    for i, (login, otype) in enumerate(todo, 1):
        result = fetch_user_location(login, otype)
        rows.append(result)

        if i % 100 == 0 or i == total:
            print(f"  [{i}/{total}] fetched — last: {login} → "
                  f"'{result['location'][:40]}'")
            # Checkpoint save every 500
            if i % 500 == 0 or i == total:
                _save(rows, out_path)

        time.sleep(0.8)  # ~72 req/min, well under 5000/hr limit

    _save(rows, out_path)

    # Summary
    locs = [r for r in rows if r["location"].strip()]
    print(f"\n📊 Summary: {len(rows)} owners fetched")
    print(f"   With location: {len(locs)} ({100*len(locs)/max(len(rows),1):.1f}%)")
    print(f"   Without location: {len(rows) - len(locs)}")


def _save(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["login", "location", "name", "company", "bio", "type", "status"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


if __name__ == "__main__":
    main()


---
## Step 5/13 — HF 作者地理位置

**原 `step3a_hf_fetch_user_locations.py`**

源文件：`ELSE/scripts/step_05_hf_author_locations.py`

执行：`import step_05_hf_author_locations as _m; _m.main()` 或在终端 `python ELSE/scripts/step_05_hf_author_locations.py`。


In [ ]:
"""
Step 3a (HF) – Resolve a "raw_location" string for every HF author /
organisation that owns a prominent HF project.

Why this is non-trivial
-----------------------
The Hugging Face Hub API does NOT expose a `location` field on its
user / organisation overview endpoints (verified 2026-04). The HTML
profile page also does not embed it as JSON. We therefore have to
recover author location from external evidence:

  1.  same-name lookup in `github_owner_locations.csv`
      (case-insensitive). Many individuals use the same handle on both
      platforms.
  2.  a small hand-curated dictionary for the highest-volume HF orgs
      (covers ~30% of HF prominent projects with very high precision).
  3.  GitHub API `GET /users/{login}` for the remaining HF authors
      that look like they could be a single person / org on GitHub.
      Skipped automatically when GITHUB_TOKEN is unset or when the
      caller passes --no-github-api.

Output
------
data/raw/huggingface/hf_author_locations.csv with columns:
    author, entity_type, raw_location, source, status, fetched_at

Where `source` ∈ {cached_github, manual, github_api, unmatched} so
downstream stages can reason about confidence.
"""

import argparse
import csv
import os
import sys
import time
from pathlib import Path

import pandas as pd
import requests

from config import DATA_RAW, DATA_PROCESSED, GITHUB_TOKEN

MASTER = DATA_PROCESSED / "prominent_projects_master.csv"
GH_LOC = DATA_RAW / "github" / "github_owner_locations.csv"
OUT    = DATA_RAW / "huggingface" / "hf_author_locations.csv"

GH_API_USER = "https://api.github.com/users/{}"
HEADERS = {
    "User-Agent": "casa0006-dsss/1.0",
    "Accept": "application/vnd.github+json",
}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

REQUEST_TIMEOUT = 15
SLEEP_BETWEEN   = 0.05

# ── Manual dictionary ─────────────────────────────────────────────────────────
# Hand-curated locations for the highest-volume HF authors.  Locations are
# written in the same free-form style as GitHub's `location` field so that
# step3b's cleaner / matcher accepts them without modification.
#
# Locations are written using the *closest known top-150 metro* rather
# than the literal head-office city, so that step3b's matcher can map
# them to a city in the curated city list. For example, "Menlo Park"
# and "Cupertino" become "San Francisco, CA, USA" (same Bay Area FUA);
# "Seongnam" becomes "Seoul, South Korea"; "Darmstadt" becomes
# "Frankfurt, Germany"; etc. This is consistent with the FUA-level
# spatial unit used elsewhere in the project.
#
MANUAL_HF_LOCATIONS = {
    # ── corporate research labs (US) ───────────────────────────────────────
    "google":              "San Francisco, CA, USA",
    "google-bert":         "San Francisco, CA, USA",
    "google-t5":           "San Francisco, CA, USA",
    "google-research":     "San Francisco, CA, USA",
    "googlefonts":         "San Francisco, CA, USA",
    "facebook":            "San Francisco, CA, USA",
    "facebookresearch":    "San Francisco, CA, USA",
    "FacebookAI":          "San Francisco, CA, USA",
    "meta-llama":          "San Francisco, CA, USA",
    "meta-music":          "San Francisco, CA, USA",
    "microsoft":           "Redmond, WA, USA",
    "Microsoft":           "Redmond, WA, USA",
    "nvidia":              "San Francisco, CA, USA",
    "NVIDIA":              "San Francisco, CA, USA",
    "intel":               "San Francisco, CA, USA",
    "Intel":               "San Francisco, CA, USA",
    "ibm":                 "New York, NY, USA",
    "IBM":                 "New York, NY, USA",
    "ibm-granite":         "New York, NY, USA",
    "amazon":              "Seattle, WA, USA",
    "AmazonScience":       "Seattle, WA, USA",
    "salesforce":          "San Francisco, CA, USA",
    "Salesforce":          "San Francisco, CA, USA",
    "apple":               "San Francisco, CA, USA",
    "Apple":               "San Francisco, CA, USA",
    "openai":              "San Francisco, CA, USA",
    "OpenAI":              "San Francisco, CA, USA",
    "anthropic":           "San Francisco, CA, USA",
    "Anthropic":           "San Francisco, CA, USA",
    "huggingface":         "New York, NY, USA",
    "HuggingFaceH4":       "New York, NY, USA",
    "huggingface-projects":"New York, NY, USA",

    # ── AI startups ────────────────────────────────────────────────────────
    "stabilityai":         "London, UK",
    "mistralai":           "Paris, France",
    "kyutai":              "Paris, France",
    "black-forest-labs":   "Frankfurt, Germany",
    "cohereai":            "Toronto, Canada",
    "CohereForAI":         "Toronto, Canada",
    "cohere":              "Toronto, Canada",
    "ai21labs":            "Tel Aviv, Israel",
    "ai21":                "Tel Aviv, Israel",
    "togethercomputer":    "San Francisco, CA, USA",
    "perplexity-ai":       "San Francisco, CA, USA",
    "deepseek-ai":         "Hangzhou, China",
    "Qwen":                "Hangzhou, China",
    "alibaba":             "Hangzhou, China",
    "Alibaba-NLP":         "Hangzhou, China",
    "baichuan-inc":        "Beijing, China",
    "baichuanai":          "Beijing, China",
    "01-ai":               "Beijing, China",
    "MoonshotAI":          "Beijing, China",
    "tencent":             "Shenzhen, China",
    "Tencent":             "Shenzhen, China",
    "internlm":            "Shanghai, China",
    "OpenGVLab":           "Shanghai, China",
    "shanghai-ai-lab":     "Shanghai, China",
    "ZhipuAI":             "Beijing, China",
    "THUDM":               "Beijing, China",
    "BAAI":                "Beijing, China",
    "IDEA-CCNL":           "Shenzhen, China",
    "IDEA-Research":       "Shenzhen, China",
    "fnlp":                "Shanghai, China",
    "FreedomIntelligence": "Shenzhen, China",
    "lmsys":               "San Francisco, CA, USA",
    "berkeley-nest":       "San Francisco, CA, USA",
    "stanfordnlp":         "San Francisco, CA, USA",
    "allenai":             "Seattle, WA, USA",
    "AllenAI":             "Seattle, WA, USA",
    "EleutherAI":          "New York, NY, USA",
    "bigscience":          "Paris, France",
    "bigcode":             "Paris, France",
    "Helsinki-NLP":        "Helsinki, Finland",
    "sentence-transformers":"Frankfurt, Germany",
    "naver-clova-ix":      "Seoul, South Korea",
    "naver":               "Seoul, South Korea",
    "kakaobrain":          "Seoul, South Korea",
    "snunlp":              "Seoul, South Korea",
    "rinna":               "Tokyo, Japan",
    "cyberagent":          "Tokyo, Japan",
    "elyza":               "Tokyo, Japan",
    "pfnet":               "Tokyo, Japan",
    "stockmark":           "Tokyo, Japan",
    "sambanova":           "San Francisco, CA, USA",
    "snowflake":           "San Francisco, CA, USA",
    "Snowflake":           "San Francisco, CA, USA",
    "databricks":          "San Francisco, CA, USA",
    "DataBricks":          "San Francisco, CA, USA",
    "deepmind":            "London, UK",
    "DeepMind":            "London, UK",
    "Tsinghua":            "Beijing, China",
    "BUPT":                "Beijing, China",
    "PKU":                 "Beijing, China",
    "openbmb":             "Beijing, China",
    "OpenMed":             "Boston, MA, USA",
    "MedARC":              "Boston, MA, USA",
    "argilla":             "Madrid, Spain",
    "vinai":               "Hanoi, Vietnam",
    "VietAI":              "Hanoi, Vietnam",
    "skywork":             "Beijing, China",
    "MBZUAI":              "Abu Dhabi, UAE",
    "ServiceNow":          "Montreal, Canada",
    "ServiceNow-AI":       "Montreal, Canada",
    "kfkas":               "Seoul, South Korea",
    "MaziyarPanahi":       "Paris, France",
    "TheBloke":            "London, UK",
    "bartowski":           "Toronto, Canada",
    "unsloth":             "San Francisco, CA, USA",
    "lmstudio-community":  "Seattle, WA, USA",
    "mradermacher":        "Hamburg, Germany",
    "timm":                "London, UK",
    "rwightman":           "London, UK",
    "ggml-org":            "Sofia, Bulgaria",
    "openchat":            "Beijing, China",
    "BlinkDL":             "Hangzhou, China",
    "RWKV":                "Hangzhou, China",
    "OpenAccess-AI-Collective":"San Francisco, CA, USA",
    "WizardLMTeam":        "Beijing, China",
    "WizardLM":            "Beijing, China",
    "Open-Orca":           "San Francisco, CA, USA",
    "NousResearch":        "Los Angeles, CA, USA",
    "RedHatAI":            "Raleigh, NC, USA",
    "AI-MO":               "Paris, France",
    "OpenLLMSG":           "Singapore",
    "AISingapore":         "Singapore",
    "answerdotai":         "San Francisco, CA, USA",
    "AnswerDotAI":         "San Francisco, CA, USA",
    "BAAI-DCAI":           "Beijing, China",
    "DAMO-NLP-SG":         "Singapore",
    "BlackForestLabs":     "Frankfurt, Germany",
    "Felladrin":           "Sao Paulo, Brazil",
    "stable-diffusion-art":"London, UK",
    "lllyasviel":          "San Francisco, CA, USA",
    "Linaqruf":            "Jakarta, Indonesia",
    "ckpt":                "Tokyo, Japan",
    "JosephusCheung":      "Hong Kong, China",
    "yiyixu":              "San Francisco, CA, USA",
    "Salesforce-Research": "San Francisco, CA, USA",
    "ml-foundations":      "Seattle, WA, USA",
    "DiscoResearch":       "Berlin, Germany",
    "VAGOsolutions":       "Munich, Germany",
    "LeoLM":               "Munich, Germany",
    "occiglot":            "Munich, Germany",
    "jondurbin":           "Boston, MA, USA",
    "PrunaAI":             "Paris, France",
    "AIDC-AI":             "Hangzhou, China",
    "OpenBMB":             "Beijing, China",
    "ZJU-Fanlab":          "Hangzhou, China",
    "moondream":           "Seattle, WA, USA",
    "vikhyatk":            "San Francisco, CA, USA",
    "x-flux":              "Moscow, Russia",
    "yandex":              "Moscow, Russia",
    "lab-mit":             "Boston, MA, USA",
    "MIT-IBM":             "Boston, MA, USA",
    "MIT":                 "Boston, MA, USA",
    "harvard-nlp":         "Boston, MA, USA",
    "princeton-nlp":       "New York, NY, USA",
    "uw-nlp":              "Seattle, WA, USA",
    "cmu-lti":             "Pittsburgh, PA, USA",
    "ucl":                 "London, UK",
    "ox-it":               "Oxford, UK",
    "cambridgeltl":        "Cambridge, UK",
    "epfl-llm":            "Lausanne, Switzerland",
    "BSC-LT":              "Barcelona, Spain",
    "PlanTL-GOB-ES":       "Madrid, Spain",
    "deepset":             "Berlin, Germany",
    "TU-Vienna":           "Vienna, Austria",

    # ── second-pass additions (after running step3a once and inspecting
    #    the highest-volume unmatched HF authors) ──────────────────────────
    "optimum-intel-internal-testing": "San Francisco, CA, USA",
    "zai-org":                "Beijing, China",
    "multimodalart":          "New York, NY, USA",
    "trl-internal-testing":   "New York, NY, USA",
    "peft-internal-testing":  "New York, NY, USA",
    "HuggingFaceTB":          "New York, NY, USA",
    "HuggingFaceM4":          "New York, NY, USA",
    "lerobot":                "New York, NY, USA",
    "Comfy-Org":              "San Francisco, CA, USA",
    "LiquidAI":               "Boston, MA, USA",
    "tiiuae":                 "Abu Dhabi, UAE",
    "PaddlePaddle":           "Beijing, China",
    "Baidu":                  "Beijing, China",
    "BAAI-DCAI":              "Beijing, China",
    "laion":                  "Hamburg, Germany",
    "LAION":                  "Hamburg, Germany",
    "Salesforce-AI":          "San Francisco, CA, USA",
    "ServiceNow-AI":          "Montreal, Canada",
    "Snowflake-AI":           "San Mateo, CA, USA",
    "neuralmagic":            "Boston, MA, USA",
    "AdaptLLM":               "Hong Kong, China",
    "ZJU-LMS":                "Hangzhou, China",
    "FudanNLP":               "Shanghai, China",
    "Skywork":                "Beijing, China",
    "InstantX":               "Beijing, China",
    "BlinkDL_AI":             "Hangzhou, China",
    "OpenBuddy":              "Singapore",
    "OrionStarAI":            "Beijing, China",
    "shenzhi-wang":           "Beijing, China",
    "Mihaiii":                "Bucharest, Romania",
    "second-state":           "San Francisco, CA, USA",
    "second-state-coreml":    "San Francisco, CA, USA",
    "Gryphe":                 "Amsterdam, Netherlands",
    "LDJnr":                  "Toronto, Canada",
    "argmaxinc":              "San Francisco, CA, USA",
    "OuteAI":                 "Vilnius, Lithuania",
    "PygmalionAI":            "San Francisco, CA, USA",
    "MaartenGr":              "Amsterdam, Netherlands",
    "BeIR":                   "Darmstadt, Germany",
    "intfloat":               "Beijing, China",
    "moka-ai":                "Beijing, China",
    "Salesforce-research":    "San Francisco, CA, USA",
    "h2oai":                  "Mountain View, CA, USA",
    "H2OAI":                  "Mountain View, CA, USA",
    "shibing624":             "Beijing, China",
    "uer":                    "Beijing, China",
    "fnlp":                   "Shanghai, China",
    "ckiplab":                "Taipei, Taiwan",
    "uitnlp":                 "Ho Chi Minh City, Vietnam",
    "uonlp":                  "Hanoi, Vietnam",
    "AntGroup":               "Hangzhou, China",
    "ant-design":             "Hangzhou, China",
    "PKU-Alignment":          "Beijing, China",
    "PKU-YuanGroup":          "Beijing, China",
    "USC-GVL":                "Los Angeles, CA, USA",
}


def _load_existing_output(path: Path) -> dict:
    if not path.exists():
        return {}
    df = pd.read_csv(path, dtype=str).fillna("")
    return {row["author"]: dict(row) for _, row in df.iterrows()}


def _load_github_loc_index() -> dict:
    if not GH_LOC.exists():
        return {}
    df = pd.read_csv(GH_LOC, dtype=str).fillna("")
    idx = {}
    for _, row in df.iterrows():
        login = row["login"].strip()
        if not login:
            continue
        loc = row["location"].strip()
        idx[login.lower()] = {
            "login_canonical": login,
            "location": loc,
            "type": (row.get("type") or "").strip() or "User",
        }
    return idx


def _query_github_user(login: str):
    """Return (location, entity_type, status)."""
    try:
        r = requests.get(GH_API_USER.format(login), headers=HEADERS,
                         timeout=REQUEST_TIMEOUT)
    except requests.RequestException as exc:
        return ("", "unknown", f"err:{type(exc).__name__}")

    if r.status_code == 404:
        return ("", "unknown", "http:404")
    if r.status_code in (401, 403):
        # rate limit or auth issue
        remaining = r.headers.get("X-RateLimit-Remaining")
        return ("", "unknown", f"http:{r.status_code}:rl={remaining}")
    if r.status_code != 200:
        return ("", "unknown", f"http:{r.status_code}")

    try:
        d = r.json()
    except ValueError:
        return ("", "unknown", "json")

    loc = (d.get("location") or "").strip()
    kind = "organization" if d.get("type") == "Organization" else "user"
    return (loc, kind, "ok")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--no-github-api", action="store_true",
                        help="Skip the GitHub /users/{login} fallback")
    parser.add_argument("--gh-budget", type=int, default=4500,
                        help="Max GitHub /users/{login} calls per run")
    args = parser.parse_args()

    print("=" * 64)
    print("Step 3a (HF): resolve HF author / org locations")
    print("=" * 64)

    if not MASTER.exists():
        print(f"❌ Missing {MASTER}. Run step1c first.")
        return

    df = pd.read_csv(MASTER, dtype=str)
    hf = df[df["platform"] == "HuggingFace"].copy()
    hf["author"] = hf["full_id"].fillna("").str.split("/").str[0]
    authors = sorted({a for a in hf["author"] if a})

    print(f"  HF prominent rows : {len(hf)}")
    print(f"  Unique HF authors : {len(authors)}")

    gh_idx = _load_github_loc_index()
    print(f"  GH login index    : {len(gh_idx)}")
    print(f"  Manual dictionary : {len(MANUAL_HF_LOCATIONS)} entries")
    print(f"  GitHub token      : {'set' if GITHUB_TOKEN else 'NOT set (skip API)'}")

    # ── Resolve in order: cached → manual → github_api ────────────────────────
    cached = _load_existing_output(OUT)
    print(f"  Already in OUT    : {len(cached)}")

    # Stats
    n_cached_hit = 0
    n_cached_loc = 0
    n_manual = 0
    n_github_cached = 0
    n_github_api = 0
    n_unmatched = 0
    n_github_calls = 0

    rows_out = {}
    for a in authors:
        # Reuse previous run if status was 'ok' or it was 'manual'/'cached_github'
        prev = cached.get(a)
        if prev and prev.get("status") in ("ok", "manual"):
            rows_out[a] = prev
            n_cached_hit += 1
            if prev.get("raw_location"):
                n_cached_loc += 1
            continue

        # Manual dictionary first (very high precision)
        if a in MANUAL_HF_LOCATIONS:
            rows_out[a] = {
                "author": a,
                "entity_type": "organization",
                "raw_location": MANUAL_HF_LOCATIONS[a],
                "source": "manual",
                "status": "manual",
                "fetched_at": "",
            }
            n_manual += 1
            continue

        # Same-name GitHub cache lookup
        gh = gh_idx.get(a.lower())
        if gh and gh["location"]:
            rows_out[a] = {
                "author": a,
                "entity_type": "organization" if gh["type"] == "Organization" else "user",
                "raw_location": gh["location"],
                "source": "cached_github",
                "status": "ok",
                "fetched_at": "",
            }
            n_github_cached += 1
            continue

        # GitHub API fallback
        if args.no_github_api or not GITHUB_TOKEN or n_github_calls >= args.gh_budget:
            rows_out[a] = {
                "author": a,
                "entity_type": "unknown",
                "raw_location": "",
                "source": "unmatched",
                "status": "skipped",
                "fetched_at": "",
            }
            n_unmatched += 1
            continue

        loc, kind, status = _query_github_user(a)
        n_github_calls += 1
        if status == "ok" and loc:
            rows_out[a] = {
                "author": a,
                "entity_type": kind,
                "raw_location": loc,
                "source": "github_api",
                "status": "ok",
                "fetched_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            n_github_api += 1
        else:
            rows_out[a] = {
                "author": a,
                "entity_type": kind,
                "raw_location": "",
                "source": "github_api",
                "status": status,
                "fetched_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            n_unmatched += 1

        if n_github_calls % 100 == 0:
            print(f"  … github_api calls so far: {n_github_calls}")

        time.sleep(SLEEP_BETWEEN)

    # ── Write ─────────────────────────────────────────────────────────────────
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["author", "entity_type", "raw_location",
                  "source", "status", "fetched_at"]
    with open(OUT, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        for a in sorted(rows_out):
            w.writerow({k: rows_out[a].get(k, "") for k in fieldnames})

    # ── Summary ───────────────────────────────────────────────────────────────
    total = len(rows_out)
    with_loc = sum(1 for r in rows_out.values() if r["raw_location"])
    print("\n" + "=" * 64)
    print("Summary")
    print("=" * 64)
    print(f"  Total authors written       : {total}")
    print(f"    cached previous           : {n_cached_hit}")
    print(f"    resolved via manual dict  : {n_manual}")
    print(f"    resolved via GH cache     : {n_github_cached}")
    print(f"    resolved via GH API call  : {n_github_api}")
    print(f"    unmatched                 : {n_unmatched}")
    print(f"  GitHub API calls made       : {n_github_calls}")
    print(f"  → with non-empty location   : {with_loc} ({with_loc/total*100:.1f}%)")
    print(f"\n✅ Wrote → {OUT}")


if __name__ == "__main__":
    main()


---
## Step 6/13 — 清洗并映射城市

**原 `step3b_clean_and_map_locations.py`**

源文件：`ELSE/scripts/step_06_clean_map_locations.py`

执行：`import step_06_clean_map_locations as _m; _m.main()` 或在终端 `python ELSE/scripts/step_06_clean_map_locations.py`。


In [ ]:
"""
Step 3b – Clean raw location strings and map them to a unified city system.

Pipeline
--------
1. Load raw locations from GitHub owners + HF authors
2. Text normalisation (lowercase, strip emoji/symbols, expand abbreviations)
3. Filter out invalid locations (remote, earth, worldwide, etc.)
4. Match to a curated city dictionary (city → country, lat, lon)
5. Assign confidence labels (high / medium / low)
6. Save location_mapping.csv

The city dictionary is built from a bundled world-cities reference.
For geocoding fallback, we use the free Nominatim API (rate-limited).
"""

import csv
import re
import sys
import time
import unicodedata
from pathlib import Path
from typing import Optional, Dict, Tuple

import pandas as pd

from config import DATA_RAW, DATA_PROCESSED

# ── Invalid location patterns ──────────────────────────────────────────────────
INVALID_PATTERNS = [
    r"^remote",
    r"^worldwide",
    r"^earth$",
    r"^earth,",
    r"^planet earth",
    r"^internet",
    r"^global",
    r"^online",
    r"^everywhere",
    r"^anywhere",
    r"^home",
    r"^mars",
    r"^moon",
    r"^space",
    r"^heaven",
    r"^hell",
    r"^localhost",
    r"^127\.0\.0",
    r"^/dev/null",
    r"^null",
    r"^n/?a$",
    r"^none$",
    r"^unknown",
    r"^planet\s",
    r"^the\s+cloud",
    r"^metaverse",
    r"^cyberspace",
    r"^virtual",
    r"^\.$",
    r"^-$",
]
INVALID_RE = re.compile("|".join(INVALID_PATTERNS), re.IGNORECASE)

# ── Common abbreviation expansions ─────────────────────────────────────────────
ABBREVIATIONS = {
    "sf": "San Francisco",
    "sf bay area": "San Francisco",
    "bay area": "San Francisco",
    "silicon valley": "San Francisco",
    "nyc": "New York",
    "ny": "New York",
    "new york city": "New York",
    "la": "Los Angeles",
    "dc": "Washington",
    "washington dc": "Washington",
    "washington d.c.": "Washington",
    "washington, d.c.": "Washington",
    "philly": "Philadelphia",
    "hk": "Hong Kong",
    "ldn": "London",
    "bj": "Beijing",
    "sh": "Shanghai",
    "gz": "Guangzhou",
    "sz": "Shenzhen",
    "spb": "Saint Petersburg",
    "st. petersburg": "Saint Petersburg",
    "st petersburg": "Saint Petersburg",
    "ist": "Istanbul",
    "cdmx": "Mexico City",
    "mexico city": "Mexico City",
    "ciudad de mexico": "Mexico City",
    "mumbai": "Mumbai",
    "bombay": "Mumbai",
    "bangalore": "Bengaluru",
    "bengaluru": "Bengaluru",
    "bangalore, india": "Bengaluru",
    "calcutta": "Kolkata",
    "madras": "Chennai",
    "saigon": "Ho Chi Minh City",
    "são paulo": "Sao Paulo",
    "sao paulo": "Sao Paulo",
    "rio de janeiro": "Rio de Janeiro",
    "muc": "Munich",
    "münchen": "Munich",
    "munich": "Munich",
    "köln": "Cologne",
    "cologne": "Cologne",
    "praha": "Prague",
    "wien": "Vienna",
    "zürich": "Zurich",
    "zurich": "Zurich",
    "genève": "Geneva",
    "montreal": "Montreal",
    "montréal": "Montreal",
    "tky": "Tokyo",
    "seoul": "Seoul",
    "taipei": "Taipei",
    "sgp": "Singapore",
}

# ── Curated top-200 global cities dictionary ───────────────────────────────────
# city_name_lower → (canonical_name, country, lat, lon)
CITY_DICT: Dict[str, Tuple[str, str, float, float]] = {}


def _build_city_dict():
    """Build a lookup dictionary from the bundled cities reference CSV,
    or fall back to a hardcoded top-200 list."""
    global CITY_DICT

    ref_path = DATA_RAW / "city_attributes" / "world_cities_reference.csv"
    if ref_path.exists():
        df = pd.read_csv(ref_path, dtype=str)
        for _, row in df.iterrows():
            key = str(row.get("city", "")).strip().lower()
            if key:
                CITY_DICT[key] = (
                    str(row.get("city", "")),
                    str(row.get("country", "")),
                    float(row.get("lat", 0)),
                    float(row.get("lon", 0)),
                )
        print(f"  Loaded {len(CITY_DICT)} cities from {ref_path}")
        return

    # Hardcoded top cities (abbreviated for space; covers major tech hubs)
    top_cities = [
        ("San Francisco", "United States", 37.7749, -122.4194),
        ("New York", "United States", 40.7128, -74.0060),
        ("Los Angeles", "United States", 34.0522, -118.2437),
        ("Seattle", "United States", 47.6062, -122.3321),
        ("Boston", "United States", 42.3601, -71.0589),
        ("Chicago", "United States", 41.8781, -87.6298),
        ("Austin", "United States", 30.2672, -97.7431),
        ("Washington", "United States", 38.9072, -77.0369),
        ("Atlanta", "United States", 33.7490, -84.3880),
        ("Denver", "United States", 39.7392, -104.9903),
        ("San Diego", "United States", 32.7157, -117.1611),
        ("Portland", "United States", 45.5152, -122.6784),
        ("San Jose", "United States", 37.3382, -121.8863),
        ("Philadelphia", "United States", 39.9526, -75.1652),
        ("Dallas", "United States", 32.7767, -96.7970),
        ("Houston", "United States", 29.7604, -95.3698),
        ("Miami", "United States", 25.7617, -80.1918),
        ("Minneapolis", "United States", 44.9778, -93.2650),
        ("Pittsburgh", "United States", 40.4406, -79.9959),
        ("Raleigh", "United States", 35.7796, -78.6382),
        ("Salt Lake City", "United States", 40.7608, -111.8910),
        ("Phoenix", "United States", 33.4484, -112.0740),
        ("Detroit", "United States", 42.3314, -83.0458),
        ("Ann Arbor", "United States", 42.2808, -83.7430),
        ("Boulder", "United States", 40.0150, -105.2705),
        ("Palo Alto", "United States", 37.4419, -122.1430),
        ("Mountain View", "United States", 37.3861, -122.0839),
        ("Sunnyvale", "United States", 37.3688, -122.0363),
        ("Cupertino", "United States", 37.3230, -122.0322),
        ("Redmond", "United States", 47.6740, -122.1215),

        ("London", "United Kingdom", 51.5074, -0.1278),
        ("Cambridge", "United Kingdom", 52.2053, 0.1218),
        ("Oxford", "United Kingdom", 51.7520, -1.2577),
        ("Edinburgh", "United Kingdom", 55.9533, -3.1883),
        ("Manchester", "United Kingdom", 53.4808, -2.2426),
        ("Bristol", "United Kingdom", 51.4545, -2.5879),

        ("Paris", "France", 48.8566, 2.3522),
        ("Lyon", "France", 45.7640, 4.8357),
        ("Toulouse", "France", 43.6047, 1.4442),
        ("Grenoble", "France", 45.1885, 5.7245),

        ("Berlin", "Germany", 52.5200, 13.4050),
        ("Munich", "Germany", 48.1351, 11.5820),
        ("Hamburg", "Germany", 53.5511, 9.9937),
        ("Frankfurt", "Germany", 50.1109, 8.6821),
        ("Cologne", "Germany", 50.9375, 6.9603),
        ("Stuttgart", "Germany", 48.7758, 9.1829),
        ("Heidelberg", "Germany", 49.3988, 8.6724),

        ("Amsterdam", "Netherlands", 52.3676, 4.9041),
        ("Rotterdam", "Netherlands", 51.9244, 4.4777),
        ("Delft", "Netherlands", 52.0116, 4.3571),
        ("Eindhoven", "Netherlands", 51.4416, 5.4697),

        ("Zurich", "Switzerland", 47.3769, 8.5417),
        ("Geneva", "Switzerland", 46.2044, 6.1432),
        ("Lausanne", "Switzerland", 46.5197, 6.6323),

        ("Stockholm", "Sweden", 59.3293, 18.0686),
        ("Gothenburg", "Sweden", 57.7089, 11.9746),
        ("Copenhagen", "Denmark", 55.6761, 12.5683),
        ("Helsinki", "Finland", 60.1699, 24.9384),
        ("Oslo", "Norway", 59.9139, 10.7522),

        ("Madrid", "Spain", 40.4168, -3.7038),
        ("Barcelona", "Spain", 41.3874, 2.1686),
        ("Lisbon", "Portugal", 38.7223, -9.1393),
        ("Rome", "Italy", 41.9028, 12.4964),
        ("Milan", "Italy", 45.4642, 9.1900),
        ("Turin", "Italy", 45.0703, 7.6869),

        ("Vienna", "Austria", 48.2082, 16.3738),
        ("Prague", "Czech Republic", 50.0755, 14.4378),
        ("Warsaw", "Poland", 52.2297, 21.0122),
        ("Krakow", "Poland", 50.0647, 19.9450),
        ("Budapest", "Hungary", 47.4979, 19.0402),
        ("Bucharest", "Romania", 44.4268, 26.1025),
        ("Dublin", "Ireland", 53.3498, -6.2603),
        ("Brussels", "Belgium", 50.8503, 4.3517),
        ("Athens", "Greece", 37.9838, 23.7275),

        ("Moscow", "Russia", 55.7558, 37.6173),
        ("Saint Petersburg", "Russia", 59.9343, 30.3351),

        ("Istanbul", "Turkey", 41.0082, 28.9784),
        ("Ankara", "Turkey", 39.9334, 32.8597),

        ("Tel Aviv", "Israel", 32.0853, 34.7818),
        ("Jerusalem", "Israel", 31.7683, 35.2137),
        ("Haifa", "Israel", 32.7940, 34.9896),

        ("Dubai", "United Arab Emirates", 25.2048, 55.2708),
        ("Abu Dhabi", "United Arab Emirates", 24.4539, 54.3773),
        ("Riyadh", "Saudi Arabia", 24.7136, 46.6753),

        ("Beijing", "China", 39.9042, 116.4074),
        ("Shanghai", "China", 31.2304, 121.4737),
        ("Shenzhen", "China", 22.5431, 114.0579),
        ("Hangzhou", "China", 30.2741, 120.1551),
        ("Guangzhou", "China", 23.1291, 113.2644),
        ("Chengdu", "China", 30.5728, 104.0668),
        ("Nanjing", "China", 32.0603, 118.7969),
        ("Wuhan", "China", 30.5928, 114.3055),
        ("Xi'an", "China", 34.3416, 108.9398),
        ("Hefei", "China", 31.8206, 117.2272),
        ("Suzhou", "China", 31.2990, 120.5853),
        ("Dalian", "China", 38.9140, 121.6147),
        ("Tianjin", "China", 39.3434, 117.3616),
        ("Hong Kong", "China", 22.3193, 114.1694),

        ("Tokyo", "Japan", 35.6762, 139.6503),
        ("Osaka", "Japan", 34.6937, 135.5023),
        ("Kyoto", "Japan", 35.0116, 135.7681),
        ("Nagoya", "Japan", 35.1815, 136.9066),
        ("Tsukuba", "Japan", 36.0835, 140.0766),

        ("Seoul", "South Korea", 37.5665, 126.9780),
        ("Busan", "South Korea", 35.1796, 129.0756),
        ("Daejeon", "South Korea", 36.3504, 127.3845),

        ("Taipei", "Taiwan", 25.0330, 121.5654),
        ("Hsinchu", "Taiwan", 24.8138, 120.9675),

        ("Singapore", "Singapore", 1.3521, 103.8198),

        ("Bengaluru", "India", 12.9716, 77.5946),
        ("Mumbai", "India", 19.0760, 72.8777),
        ("New Delhi", "India", 28.6139, 77.2090),
        ("Delhi", "India", 28.7041, 77.1025),
        ("Hyderabad", "India", 17.3850, 78.4867),
        ("Chennai", "India", 13.0827, 80.2707),
        ("Pune", "India", 18.5204, 73.8567),
        ("Kolkata", "India", 22.5726, 88.3639),

        ("Jakarta", "Indonesia", -6.2088, 106.8456),
        ("Bangkok", "Thailand", 13.7563, 100.5018),
        ("Kuala Lumpur", "Malaysia", 3.1390, 101.6869),
        ("Ho Chi Minh City", "Vietnam", 10.8231, 106.6297),
        ("Hanoi", "Vietnam", 21.0278, 105.8342),
        ("Manila", "Philippines", 14.5995, 120.9842),

        ("Sydney", "Australia", -33.8688, 151.2093),
        ("Melbourne", "Australia", -37.8136, 144.9631),
        ("Brisbane", "Australia", -27.4698, 153.0251),
        ("Perth", "Australia", -31.9505, 115.8605),
        ("Canberra", "Australia", -35.2809, 149.1300),
        ("Auckland", "New Zealand", -36.8485, 174.7633),

        ("Toronto", "Canada", 43.6532, -79.3832),
        ("Vancouver", "Canada", 49.2827, -123.1207),
        ("Montreal", "Canada", 45.5017, -73.5673),
        ("Ottawa", "Canada", 45.4215, -75.6972),
        ("Calgary", "Canada", 51.0447, -114.0719),
        ("Edmonton", "Canada", 53.5461, -113.4938),
        ("Waterloo", "Canada", 43.4643, -80.5204),

        ("Sao Paulo", "Brazil", -23.5505, -46.6333),
        ("Rio de Janeiro", "Brazil", -22.9068, -43.1729),
        ("Mexico City", "Mexico", 19.4326, -99.1332),
        ("Buenos Aires", "Argentina", -34.6037, -58.3816),
        ("Santiago", "Chile", -33.4489, -70.6693),
        ("Bogota", "Colombia", 4.7110, -74.0721),
        ("Lima", "Peru", -12.0464, -77.0428),

        ("Cairo", "Egypt", 30.0444, 31.2357),
        ("Nairobi", "Kenya", -1.2921, 36.8219),
        ("Lagos", "Nigeria", 6.5244, 3.3792),
        ("Cape Town", "South Africa", -33.9249, 18.4241),
        ("Johannesburg", "South Africa", -26.2041, 28.0473),
    ]

    for city, country, lat, lon in top_cities:
        CITY_DICT[city.lower()] = (city, country, lat, lon)

    print(f"  Built hardcoded city dictionary: {len(CITY_DICT)} cities")


# ── Text cleaning ──────────────────────────────────────────────────────────────

def _strip_emoji(text):
    """Remove emoji and other non-Latin/CJK symbols."""
    return "".join(
        c for c in text
        if unicodedata.category(c)[0] in ("L", "M", "N", "P", "Z")
    )


def clean_location(raw: str) -> Optional[str]:
    """Normalise a raw location string. Returns None if invalid."""
    if not raw or not raw.strip():
        return None

    text = raw.strip()
    text = _strip_emoji(text)
    text = text.strip(" ,.-;:!?")

    if len(text) < 2:
        return None

    if INVALID_RE.search(text):
        return None

    return text


def match_city(cleaned: str) -> Optional[Tuple[str, str, float, float, str]]:
    """
    Try to match a cleaned location string to the city dictionary.
    Returns (city, country, lat, lon, confidence) or None.
    """
    if not cleaned:
        return None

    lower = cleaned.lower().strip()

    # 1. Direct abbreviation lookup
    if lower in ABBREVIATIONS:
        expanded = ABBREVIATIONS[lower].lower()
        if expanded in CITY_DICT:
            city, country, lat, lon = CITY_DICT[expanded]
            return (city, country, lat, lon, "high")

    # 2. Exact match
    if lower in CITY_DICT:
        city, country, lat, lon = CITY_DICT[lower]
        return (city, country, lat, lon, "high")

    # 3. Try first part before comma (e.g. "London, UK" → "london")
    parts = [p.strip() for p in lower.split(",")]
    for part in parts:
        part_clean = part.strip()
        if part_clean in ABBREVIATIONS:
            part_clean = ABBREVIATIONS[part_clean].lower()
        if part_clean in CITY_DICT:
            city, country, lat, lon = CITY_DICT[part_clean]
            return (city, country, lat, lon, "high")

    # 4. Try parts split by "/" or "&"
    for sep in ["/", "&", " - ", " and "]:
        if sep in lower:
            sub_parts = [p.strip() for p in lower.split(sep)]
            for sp in sub_parts:
                if sp in ABBREVIATIONS:
                    sp = ABBREVIATIONS[sp].lower()
                if sp in CITY_DICT:
                    city, country, lat, lon = CITY_DICT[sp]
                    return (city, country, lat, lon, "medium")

    # 5. Substring search: check if any city name appears in the text
    for key, (city, country, lat, lon) in CITY_DICT.items():
        if len(key) >= 4 and key in lower:
            return (city, country, lat, lon, "medium")

    return None


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("Step 3b: Clean locations & map to cities")
    print("=" * 60)

    _build_city_dict()

    # --- Collect raw locations ---
    records = []

    # GitHub owners
    gh_loc_path = DATA_RAW / "github" / "github_owner_locations.csv"
    if gh_loc_path.exists():
        gh = pd.read_csv(gh_loc_path, dtype=str).fillna("")
        for _, row in gh.iterrows():
            records.append({
                "entity_id": row["login"],
                "entity_type": "github_" + (row.get("type", "User")).lower(),
                "raw_location": row["location"],
                "platform": "GitHub",
            })
        print(f"  GitHub owner records: {len(gh)}")

    # HF authors – read locations resolved by step3a_hf_fetch_user_locations.py
    # (HF API does not expose location, so we rely on a hand-curated dictionary
    # for top orgs + same-name GitHub matching + GitHub API fallback).
    hf_loc_path = DATA_RAW / "huggingface" / "hf_author_locations.csv"
    if hf_loc_path.exists():
        hf_loc = pd.read_csv(hf_loc_path, dtype=str).fillna("")
        for _, row in hf_loc.iterrows():
            author = row["author"]
            if not author:
                continue
            entity_kind = row.get("entity_type", "") or "user"
            records.append({
                "entity_id": author,
                "entity_type": "hf_" + entity_kind,
                "raw_location": row.get("raw_location", ""),
                "platform": "HuggingFace",
            })
        print(f"  HF author records (from hf_author_locations.csv): "
              f"{len(hf_loc)}")
    else:
        # Fallback: emit empty raw_location so step3b is still runnable
        # before step3a has been executed. Will yield zero matched cities.
        hf_path = DATA_RAW / "huggingface" / "hf_candidates.csv"
        if hf_path.exists():
            hf = pd.read_csv(hf_path, dtype=str).fillna("")
            hf_prom = hf[hf["prominent_flag"] == "1"]
            hf_authors = hf_prom[["author"]].drop_duplicates()
            for _, row in hf_authors.iterrows():
                author = row["author"]
                if author:
                    records.append({
                        "entity_id": author,
                        "entity_type": "hf_author",
                        "raw_location": "",
                        "platform": "HuggingFace",
                    })
            print(f"  ⚠️  No hf_author_locations.csv found; "
                  f"falling back to empty raw_location for "
                  f"{len(hf_authors)} HF authors. "
                  f"Run step3a_hf_fetch_user_locations.py first.")

    print(f"  Total raw records: {len(records)}\n")

    # --- Clean and match ---
    results = []
    stats = {"total": 0, "empty": 0, "invalid": 0, "matched": 0, "unmatched": 0}

    for rec in records:
        stats["total"] += 1
        raw = rec["raw_location"]

        if not raw.strip():
            stats["empty"] += 1
            results.append({**rec, "cleaned_location": "",
                            "matched_city": "", "country": "",
                            "lat": "", "lon": "", "confidence": "none"})
            continue

        cleaned = clean_location(raw)
        if cleaned is None:
            stats["invalid"] += 1
            results.append({**rec, "cleaned_location": raw,
                            "matched_city": "", "country": "",
                            "lat": "", "lon": "", "confidence": "none"})
            continue

        match = match_city(cleaned)
        if match:
            city, country, lat, lon, conf = match
            stats["matched"] += 1
            results.append({**rec, "cleaned_location": cleaned,
                            "matched_city": city, "country": country,
                            "lat": lat, "lon": lon, "confidence": conf})
        else:
            stats["unmatched"] += 1
            results.append({**rec, "cleaned_location": cleaned,
                            "matched_city": "", "country": "",
                            "lat": "", "lon": "", "confidence": "low"})

    # --- Save ---
    out_path = DATA_PROCESSED / "location_mapping.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_out = pd.DataFrame(results)
    df_out.to_csv(out_path, index=False)
    print(f"✅ Saved {len(results)} rows → {out_path}")

    # --- Summary ---
    print(f"\n📊 Location matching summary:")
    print(f"   Total:     {stats['total']}")
    print(f"   Empty:     {stats['empty']}")
    print(f"   Invalid:   {stats['invalid']}")
    print(f"   Matched:   {stats['matched']}")
    print(f"   Unmatched: {stats['unmatched']}")

    if stats["matched"] > 0:
        df_matched = df_out[df_out["matched_city"] != ""]
        city_counts = df_matched["matched_city"].value_counts()
        print(f"\n📊 Top 20 cities by entity count:")
        print(city_counts.head(20).to_string())

        conf_dist = df_matched["confidence"].value_counts()
        print(f"\n📊 Confidence distribution:")
        print(conf_dist.to_string())


if __name__ == "__main__":
    main()


---
## Step 7/13 — Nominatim 补地理编码

**原 `step3c_geocode_unmatched.py`**

源文件：`ELSE/scripts/step_07_geocode_unmatched.py`

执行：`import step_07_geocode_unmatched as _m; _m.main()` 或在终端 `python ELSE/scripts/step_07_geocode_unmatched.py`。


In [ ]:
"""
Step 3c – Geocode unmatched locations via Nominatim (free, 1 req/sec).

Takes the `location_mapping.csv` from step3b, finds rows where
matched_city is empty but cleaned_location is not, and tries to
geocode them with OpenStreetMap Nominatim.

Then re-matches geocoded results to the nearest city in our dictionary.

Output: updates data/processed/location_mapping.csv in place.
"""

import sys
import time
from pathlib import Path

import pandas as pd
import requests

from config import DATA_PROCESSED

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
HEADERS = {"User-Agent": "CASA0006-Research/1.0 (academic project)"}

MAX_UNIQUE_QUERIES = 2000

# ── Non-English → English normalisation maps ───────────────────────────────────
COUNTRY_NORM = {
    "中国": "China", "中华人民共和国": "China",
    "日本": "Japan",
    "대한민국": "South Korea", "한국": "South Korea",
    "Deutschland": "Germany", "Bundesrepublik Deutschland": "Germany",
    "République française": "France",
    "Italia": "Italy",
    "España": "Spain",
    "Россия": "Russia", "Российская Федерация": "Russia",
    "Brasil": "Brazil",
    "Türkiye": "Turkey",
    "Ελλάς": "Greece",
    "Österreich": "Austria",
    "Schweiz/Suisse/Svizzera/Svizra": "Switzerland",
    "Suisse": "Switzerland", "Schweiz": "Switzerland",
    "Nederland": "Netherlands",
    "Polska": "Poland",
    "Česko": "Czech Republic", "Česká republika": "Czech Republic",
    "Magyarország": "Hungary",
    "România": "Romania",
    "Україна": "Ukraine",
    "ישראל": "Israel",
    "ایران": "Iran",
    "مصر": "Egypt",
    "السعودية": "Saudi Arabia",
    "الإمارات العربية المتحدة": "United Arab Emirates",
    "台灣": "Taiwan", "臺灣": "Taiwan",
    "香港": "China",
    "Việt Nam": "Vietnam",
    "ประเทศไทย": "Thailand",
    "Suomi / Finland": "Finland", "Suomi": "Finland",
    "Sverige": "Sweden",
    "Norge": "Norway",
    "Danmark": "Denmark",
    "Éire / Ireland": "Ireland", "Éire": "Ireland",
    "Belgique - België": "Belgium", "België / Belgique / Belgien": "Belgium",
    "México": "Mexico",
    "Perú": "Peru",
    "भारत": "India",
    "Pilipinas": "Philippines",
    "Ísland": "Iceland",
    "Lietuva": "Lithuania", "Latvija": "Latvia", "Eesti": "Estonia",
    "Slovensko": "Slovakia", "Slovenija": "Slovenia",
    "Hrvatska": "Croatia", "Србија": "Serbia", "България": "Bulgaria",
    "საქართველო": "Georgia",
    "Қазақстан": "Kazakhstan", "Беларусь": "Belarus",
    "پاکستان": "Pakistan",
    "नेपाल": "Nepal", "नेपाल": "Nepal",
    "বাংলাদেশ": "Bangladesh",
    "Maroc ⵍⵎⵖⵔⵉⴱ المغرب": "Morocco",
    "ኢትዮጵያ": "Ethiopia",
    "မြန်မာ": "Myanmar",
    "Crna Gora / Црна Гора": "Montenegro",
    "Oʻzbekiston": "Uzbekistan",
    "العراق": "Iraq",
    "Κύπρος - Kıbrıs": "Cyprus",
}

CITY_NORM = {
    "成都市": "Chengdu", "昌平区": "Beijing", "深圳市": "Shenzhen",
    "广州市": "Guangzhou", "武汉市": "Wuhan", "南京市": "Nanjing",
    "西安市": "Xi'an", "合肥市": "Hefei", "苏州市": "Suzhou",
    "大连市": "Dalian", "天津市": "Tianjin", "杭州市": "Hangzhou",
    "上海市": "Shanghai", "北京市": "Beijing", "重庆市": "Chongqing",
    "长沙市": "Changsha", "哈尔滨市": "Harbin", "济南市": "Jinan",
    "青岛市": "Qingdao", "郑州市": "Zhengzhou",
    "شهر تهران": "Tehran", "القاهرة": "Cairo",
    "서울": "Seoul",
    "東京都": "Tokyo", "大阪市": "Osaka", "京都市": "Kyoto",
    "台北": "Taipei", "新竹": "Hsinchu",
    "Москва": "Moscow", "Санкт-Петербург": "Saint Petersburg",
    "München": "Munich", "Köln": "Cologne",
    "Zürich": "Zurich", "Genève": "Geneva",
    "Praha": "Prague", "Wien": "Vienna",
    "Αθήνα": "Athens", "İstanbul": "Istanbul",
    "لاہور": "Lahore",
    "София": "Sofia",
    "کراچی ڈویژن": "Karachi",
    "काठमाडौँ महानगरपालिका": "Kathmandu",
    "اسلام آباد": "Islamabad",
    "Львів": "Lviv",
    "广东省": "Guangzhou",
    "沈阳市": "Shenyang",
    "昆明市": "Kunming",
    "Wrocław": "Wroclaw",
    "Thành phố Hồ Chí Minh": "Ho Chi Minh City",
    "Thành phố Hà Nội": "Hanoi",
    "København": "Copenhagen",
    "Београд": "Belgrade",
    "珠海市": "Zhuhai",
    "አዲስ አበባ أديس أبابا": "Addis Ababa",
    "长春市": "Changchun",
    "Nürnberg": "Nuremberg",
    "دبي": "Dubai",
    "厦门市": "Xiamen",
}


def geocode_nominatim(query):
    """Return (city, country, lat, lon) or None."""
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "addressdetails": 1,
        "accept-language": "en",
    }
    try:
        resp = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            return None
        results = resp.json()
        if not results:
            return None
        r = results[0]
        addr = r.get("address", {})
        city = (addr.get("city")
                or addr.get("town")
                or addr.get("village")
                or addr.get("state")
                or addr.get("county")
                or "")
        country = addr.get("country", "")
        lat = float(r.get("lat", 0))
        lon = float(r.get("lon", 0))
        if city:
            return (city, country, lat, lon)
    except Exception:
        pass
    return None


def main():
    print("=" * 60)
    print("Step 3c: Geocode unmatched locations (Nominatim)")
    print("=" * 60)

    path = DATA_PROCESSED / "location_mapping.csv"
    df = pd.read_csv(path, dtype=str).fillna("")

    unmatched = df[(df["matched_city"] == "") & (df["cleaned_location"] != "")]
    print(f"  Total rows: {len(df)}")
    print(f"  Unmatched with cleaned_location: {len(unmatched)}")

    if len(unmatched) == 0:
        print("  Nothing to geocode.")
        return

    # Get unique queries, sorted by frequency
    query_counts = unmatched["cleaned_location"].value_counts()
    unique_queries = query_counts.head(MAX_UNIQUE_QUERIES).index.tolist()
    print(f"  Unique queries to geocode: {len(unique_queries)}\n")

    cache = {}
    success = 0
    for i, q in enumerate(unique_queries, 1):
        result = geocode_nominatim(q)
        cache[q] = result
        status = f"→ {result[0]}, {result[1]}" if result else "→ (no result)"
        if result:
            success += 1
        if i % 50 == 0 or i <= 5:
            print(f"  [{i}/{len(unique_queries)}] '{q[:40]}' {status}")
        time.sleep(1.1)  # Nominatim rate limit

    print(f"\n  Geocoded successfully: {success}/{len(unique_queries)}")

    # Apply geocoding results and normalise non-English names in a single pass
    updated, fixed_c, fixed_m = 0, 0, 0
    for idx, row in df.iterrows():
        city = row["matched_city"]
        country = row["country"]

        if city == "" and row["cleaned_location"] in cache:
            result = cache[row["cleaned_location"]]
            if result:
                city, country, lat, lon = result
                df.at[idx, "matched_city"] = city
                df.at[idx, "country"] = country
                df.at[idx, "lat"] = str(lat)
                df.at[idx, "lon"] = str(lon)
                df.at[idx, "confidence"] = "medium"
                updated += 1

        if country in COUNTRY_NORM:
            df.at[idx, "country"] = COUNTRY_NORM[country]
            fixed_c += 1
        if city in CITY_NORM:
            df.at[idx, "matched_city"] = CITY_NORM[city]
            fixed_m += 1

    print(f"\n  Normalised {fixed_c} country names, {fixed_m} city names to English")

    df.to_csv(path, index=False)
    print(f"✅ Updated {updated} rows, saved → {path}")

    # Summary
    matched = df[df["matched_city"] != ""]
    print(f"\n📊 After geocoding + normalisation:")
    print(f"   Matched:   {len(matched)} / {len(df)}")
    print(f"   Unmatched: {len(df) - len(matched)}")


if __name__ == "__main__":
    main()


---
## Step 8/13 — 构建城市列表

**原 `step3d_build_city_list.py`**

源文件：`ELSE/scripts/step_08_build_city_list.py`

执行：`import step_08_build_city_list as _m; _m.main()` 或在终端 `python ELSE/scripts/step_08_build_city_list.py`。


In [ ]:
"""
Step 3d – Build the final curated city list (100-150 high-confidence cities).

1. Load location_mapping.csv
2. Keep only high + medium confidence matches
3. Aggregate: count how many prominent-project owners are in each city
4. Rank cities and select top 100-150
5. Output: data/processed/city_list.csv

This city list becomes the spatial backbone for all downstream analysis.
"""

import sys
from pathlib import Path

import pandas as pd

from config import DATA_PROCESSED

TARGET_CITY_COUNT = 150  # upper bound; may keep fewer if not enough


def main():
    print("=" * 60)
    print("Step 3d: Build curated city list")
    print("=" * 60)

    loc_path = DATA_PROCESSED / "location_mapping.csv"
    df = pd.read_csv(loc_path, dtype=str).fillna("")

    # Keep only high/medium confidence
    confident = df[df["confidence"].isin(["high", "medium"])].copy()
    print(f"  Total location records: {len(df)}")
    print(f"  High/medium confidence: {len(confident)}")

    # Include both GitHub and Hugging Face entities. HF authors used to be
    # excluded (no location was available) but step3a_hf_fetch_user_locations.py
    # now resolves locations via manual dictionary + same-name GitHub lookup,
    # so HF authors can legitimately contribute to the city list.
    confident = confident[confident["platform"].isin(["GitHub", "HuggingFace"])]
    print(f"  GitHub + HF high/medium: {len(confident)}")
    print(f"    GitHub: {(confident['platform']=='GitHub').sum()}")
    print(f"    HF    : {(confident['platform']=='HuggingFace').sum()}")

    # Filter out suspicious city names (2 chars or fewer, like "Eu", "Us")
    confident = confident[confident["matched_city"].str.len() > 2]

    # Convert lat/lon to numeric before aggregation
    confident["lat"] = pd.to_numeric(confident["lat"], errors="coerce")
    confident["lon"] = pd.to_numeric(confident["lon"], errors="coerce")

    # Group by city name + country ONLY (not lat/lon) to merge duplicates
    city_counts = (
        confident.groupby(["matched_city", "country"])
        .agg(
            entity_count=("entity_id", "nunique"),
            lat=("lat", "mean"),
            lon=("lon", "mean"),
        )
        .reset_index()
        .sort_values("entity_count", ascending=False)
    )

    print(f"\n  Unique cities found: {len(city_counts)}")

    # Select top cities
    selected = city_counts.head(TARGET_CITY_COUNT).copy()
    selected = selected.reset_index(drop=True)
    selected.index = selected.index + 1
    selected.index.name = "rank"

    # Save
    out_path = DATA_PROCESSED / "city_list.csv"
    selected.to_csv(out_path)
    print(f"\n✅ Saved {len(selected)} cities → {out_path}")

    # Display top 30
    print(f"\n📊 Top 30 cities:")
    print(selected.head(30)[["matched_city", "country", "entity_count"]]
          .to_string())

    # Regional summary
    print(f"\n📊 Cities by country (top 15):")
    country_counts = selected.groupby("country").agg(
        cities=("matched_city", "count"),
        total_entities=("entity_count", "sum"),
    ).sort_values("total_entities", ascending=False)
    print(country_counts.head(15).to_string())


if __name__ == "__main__":
    main()


---
## Step 9/13 — 贡献者与参与事件

**原 `step4_fetch_contributors.py`**

源文件：`ELSE/scripts/step_09_contributors_participation.py`

执行：`import step_09_contributors_participation as _m; _m.main()` 或在终端 `python ELSE/scripts/step_09_contributors_participation.py`。


In [ ]:
"""
Step 4 – Fetch contributor data for prominent GitHub repos.

Part A: GET /repos/{owner}/{repo}/contributors  → contributor list
Part B: GET /users/{login}                      → contributor locations
Part C: GET /repos/{owner}/{repo}/commits       → commit history
         GET /repos/{owner}/{repo}/pulls         → PR history
         Build per-actor first-participation events for adoption-lag.

Outputs
-------
data/raw/github/github_repo_contributors.csv
    repo_full_name, contributor_login, contributions

data/raw/github/github_owner_locations.csv   (appended with new users)

data/raw/github/github_repo_participation_events.csv
    repo_full_name, actor_login, first_event_at, event_type

Supports resume: skips repos / users already fetched.
"""

import csv
import sys
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from config import GITHUB_TOKEN, DATA_RAW, TIME_START

HEADERS = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

_RETRY_STRATEGY = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

_thread_local = threading.local()


def _get_session():
    """Return a per-thread requests.Session (thread-safe)."""
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        s.mount("https://", HTTPAdapter(
            max_retries=_RETRY_STRATEGY,
            pool_maxsize=20,
        ))
        s.headers.update(HEADERS)
        _thread_local.session = s
    return _thread_local.session

CONTRIB_PER_PAGE = 100
MAX_CONTRIB_PAGES = 3        # up to 300 contributors per repo
LOCATION_BATCH_SAVE = 500    # checkpoint save interval


def _wait(resp):
    remaining = int(resp.headers.get("X-RateLimit-Remaining", 1))
    if remaining < 10:
        reset_ts = int(resp.headers.get("X-RateLimit-Reset", 0))
        sleep_sec = max(reset_ts - int(time.time()), 1) + 2
        print(f"  ⏳ Rate-limited ({remaining} left). Sleeping {sleep_sec}s …")
        time.sleep(sleep_sec)


# ───────────────────────────── Part A: Contributors ──────────────────────────

def _get_with_retry(url, params=None, max_retries=5):
    """GET with manual retry for ConnectionError on top of urllib3 retry."""
    session = _get_session()
    for attempt in range(max_retries):
        try:
            return session.get(url, params=params, timeout=30)
        except (requests.ConnectionError, requests.Timeout,
                requests.exceptions.ChunkedEncodingError) as exc:
            wait = min(2 ** (attempt + 1), 120)
            print(f"  ⚠️  Connection error (attempt {attempt+1}/{max_retries}): {exc!r}")
            print(f"      Retrying in {wait}s …")
            time.sleep(wait)
    return session.get(url, params=params, timeout=30)


def fetch_contributors(repo_full_name):
    """Return list of (login, contributions) for a repo."""
    results = []
    for page in range(1, MAX_CONTRIB_PAGES + 1):
        url = f"https://api.github.com/repos/{repo_full_name}/contributors"
        params = {"per_page": CONTRIB_PER_PAGE, "page": page}
        resp = _get_with_retry(url, params=params)
        _wait(resp)
        if resp.status_code == 403:
            time.sleep(60)
            resp = _get_with_retry(url, params=params)
        if resp.status_code != 200:
            break
        items = resp.json()
        if not isinstance(items, list) or not items:
            break
        for c in items:
            login = c.get("login", "")
            contribs = c.get("contributions", 0)
            if login:
                results.append((login, contribs))
        if len(items) < CONTRIB_PER_PAGE:
            break
        time.sleep(0.3)
    return results


def collect_all_contributors():
    gh_csv = DATA_RAW / "github" / "github_candidates.csv"
    df = pd.read_csv(gh_csv, dtype=str)
    prominent = df[df["prominent_flag"] == "1"]
    repos = prominent["repo_full_name"].dropna().unique().tolist()
    print(f"  Prominent repos to fetch contributors for: {len(repos)}")

    out_path = DATA_RAW / "github" / "github_repo_contributors.csv"
    done_repos = set()
    existing_rows = []
    if out_path.exists():
        ex = pd.read_csv(out_path, dtype=str)
        done_repos = set(ex["repo_full_name"].unique())
        existing_rows = ex.values.tolist()
        print(f"  Already fetched: {len(done_repos)} repos (resuming)")

    todo = [r for r in repos if r not in done_repos]
    print(f"  Remaining: {len(todo)}\n")

    rows = list(existing_rows)
    for i, repo in enumerate(todo, 1):
        contribs = fetch_contributors(repo)
        for login, cnt in contribs:
            rows.append([repo, login, str(cnt)])

        if i % 100 == 0 or i == len(todo):
            print(f"  [{i}/{len(todo)}] {repo} → {len(contribs)} contributors")
            _save_contrib(rows, out_path)

        time.sleep(0.5)

    _save_contrib(rows, out_path)
    unique_logins = set(r[1] for r in rows)
    print(f"\n✅ Contributors: {len(rows)} records, {len(unique_logins)} unique users")
    return unique_logins


def _save_contrib(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["repo_full_name", "contributor_login", "contributions"])
        w.writerows(rows)


# ──────────────────────── Part B: Contributor Locations ───────────────────────

def fetch_user_location(login):
    url = f"https://api.github.com/users/{login}"
    resp = _get_with_retry(url)
    _wait(resp)
    if resp.status_code != 200:
        return {"login": login, "location": "", "name": "",
                "company": "", "bio": "", "type": "User", "status": f"http_{resp.status_code}"}
    data = resp.json()
    return {
        "login": login,
        "location": data.get("location") or "",
        "name": data.get("name") or "",
        "company": data.get("company") or "",
        "bio": (data.get("bio") or "")[:200],
        "type": data.get("type") or "User",
        "status": "ok",
    }


LOCATION_WORKERS = 10  # concurrent threads for location fetching


def fetch_missing_locations(all_logins):
    loc_path = DATA_RAW / "github" / "github_owner_locations.csv"
    if loc_path.exists():
        existing = pd.read_csv(loc_path, dtype=str)
        done = set(existing["login"].tolist())
        rows = existing.to_dict("records")
    else:
        done = set()
        rows = []

    todo = sorted(all_logins - done)
    print(f"\n  New contributor logins to fetch location: {len(todo)}")
    if not todo:
        print("  All locations already fetched.")
        return

    fieldnames = ["login", "location", "name", "company", "bio", "type", "status"]
    lock = threading.Lock()
    completed = [0]

    def _save_locations():
        with open(loc_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(rows)

    def process_login(login):
        result = fetch_user_location(login)
        with lock:
            rows.append(result)
            completed[0] += 1
            if completed[0] % 500 == 0 or completed[0] == len(todo):
                print(f"  [{completed[0]}/{len(todo)}] last: {login} "
                      f"→ '{str(result.get('location', ''))[:40]}'")
                _save_locations()

    with ThreadPoolExecutor(max_workers=LOCATION_WORKERS) as pool:
        futures = [pool.submit(process_login, login) for login in todo]
        for f in as_completed(futures):
            try:
                f.result()
            except Exception as exc:
                print(f"  ⚠️  Location worker exception: {exc!r}")

    _save_locations()
    locs = [r for r in rows if isinstance(r.get("location", ""), str)
            and r["location"].strip()]
    print(f"\n✅ Locations: {len(rows)} total, {len(locs)} with location")


# ═══════════════════════════════════════════════════════════════════════════════
# Part C: Participation Chain (Commits API + PRs API)
#
# For each prominent repo, paginate through commits and PRs to extract
# each actor's FIRST participation event with a precise timestamp.
# All external requests are strictly serial with rate-limit checks.
# ═══════════════════════════════════════════════════════════════════════════════

MAX_COMMIT_PAGES = 50   # cap: 5 000 commits per repo
MAX_PR_PAGES = 20       # cap: 2 000 PRs per repo
CHECKPOINT_EVERY = 10   # save progress every N repos
LOG_EVERY = 50          # print detailed log every N repos


def _strict_wait(resp):
    """Rate-limit guard for serial API calls (stricter than _wait)."""
    remaining = int(resp.headers.get("X-RateLimit-Remaining", 999))
    if remaining < 50:
        reset_ts = int(resp.headers.get("X-RateLimit-Reset", 0))
        sleep_sec = max(reset_ts - int(time.time()), 1) + 3
        print(f"  ⏳ Rate limit: {remaining} remaining. "
              f"Sleeping {sleep_sec}s until reset …")
        time.sleep(sleep_sec)


def _api_get(url, params=None):
    """Single serial GET with strict rate-limit check."""
    resp = _get_with_retry(url, params=params)
    _strict_wait(resp)
    return resp


def _fetch_repo_commits(repo):
    """
    Paginate GET /repos/{owner}/{repo}/commits.
    Return {login: earliest_ISO_date} for each commit author.
    """
    author_first = {}
    url = f"https://api.github.com/repos/{repo}/commits"
    base_params = {"per_page": 100}

    for page in range(1, MAX_COMMIT_PAGES + 1):
        params = {**base_params, "page": page}
        resp = _api_get(url, params=params)
        if resp.status_code == 409:
            break  # empty repo
        if resp.status_code != 200:
            break
        commits = resp.json()
        if not isinstance(commits, list) or not commits:
            break

        for c in commits:
            gh_author = c.get("author") or {}
            login = gh_author.get("login", "")
            if not login:
                continue
            date = (c.get("commit") or {}).get("author", {}).get("date", "")
            if not date:
                continue
            if login not in author_first or date < author_first[login]:
                author_first[login] = date

        if len(commits) < 100:
            break
    return author_first


def _fetch_repo_prs(repo):
    """
    Paginate GET /repos/{owner}/{repo}/pulls (oldest first).
    Return {login: earliest_ISO_date} for each PR author.
    """
    author_first = {}
    url = f"https://api.github.com/repos/{repo}/pulls"
    base_params = {"state": "all", "sort": "created",
                   "direction": "asc", "per_page": 100}

    for page in range(1, MAX_PR_PAGES + 1):
        params = {**base_params, "page": page}
        resp = _api_get(url, params=params)
        if resp.status_code != 200:
            break
        prs = resp.json()
        if not isinstance(prs, list) or not prs:
            break

        for pr in prs:
            login = (pr.get("user") or {}).get("login", "")
            if not login:
                continue
            date = pr.get("created_at", "")
            if not date:
                continue
            if login not in author_first or date < author_first[login]:
                author_first[login] = date

        if len(prs) < 100:
            break
    return author_first


def _save_events(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["repo_full_name", "actor_login",
                     "first_event_at", "event_type"])
        w.writerows(rows)


def _build_target_repo_set():
    """
    Pre-filter: find repos that have at least one contributor in a
    target city.  Repos with no target-city contributor are skipped
    to save ~30% of API calls.  Pure local computation.
    """
    from step_06_clean_map_locations import (
        _build_city_dict, clean_location, match_city,
    )
    from config import DATA_PROCESSED

    contrib_path = DATA_RAW / "github" / "github_repo_contributors.csv"
    loc_path = DATA_RAW / "github" / "github_owner_locations.csv"
    city_path = DATA_PROCESSED / "city_list.csv"

    if not all(p.exists() for p in [contrib_path, loc_path, city_path]):
        return None  # can't filter, process all

    cities = pd.read_csv(city_path, dtype=str)
    city_set = set(cities["matched_city"].str.strip().tolist())

    loc = pd.read_csv(loc_path, dtype=str).fillna("")
    _build_city_dict()
    user_city = set()
    for _, row in loc.iterrows():
        raw = row["location"]
        if not raw.strip():
            continue
        cleaned = clean_location(raw)
        if cleaned is None:
            continue
        result = match_city(cleaned)
        if result and result[0] in city_set:
            user_city.add(row["login"])

    contrib = pd.read_csv(contrib_path, dtype=str)
    relevant = contrib[contrib["contributor_login"].isin(user_city)]
    target_repos = set(relevant["repo_full_name"].unique())

    # Also include repos whose OWNER is in a target city
    gh = pd.read_csv(DATA_RAW / "github" / "github_candidates.csv", dtype=str)
    prom = gh[gh["prominent_flag"] == "1"]
    owner_repos = prom[prom["owner_login"].isin(user_city)]["repo_full_name"]
    target_repos.update(owner_repos.tolist())

    return target_repos


def collect_participation_events(shard_id=None, shard_total=None):
    """
    For each prominent repo, build a participation chain:
      1. 'create' event from candidates CSV (owner + created_at)
      2. 'commit' events from Commits API (each author's first commit)
      3. 'pr_opened' events from PRs API (each author's first PR)

    For each (repo, actor), only the earliest event is kept.

    Optimizations:
      - Pre-filter to repos with target-city contributors (~30% fewer calls)
      - Sharding: split work across multiple processes (--shard N/M)
      - Checkpoint every CHECKPOINT_EVERY repos
      - Graceful KeyboardInterrupt handling (saves before exit)
    """
    gh_csv = DATA_RAW / "github" / "github_candidates.csv"
    df = pd.read_csv(gh_csv, dtype=str)
    prominent = df[df["prominent_flag"] == "1"].copy()
    all_repos = prominent["repo_full_name"].dropna().unique().tolist()

    # Pre-filter: only repos with target-city contributors
    print("  Pre-filtering repos with target-city contributors …")
    target_repos = _build_target_repo_set()
    if target_repos is not None:
        repos = [r for r in all_repos if r in target_repos]
        skipped = len(all_repos) - len(repos)
        print(f"  Target-city repos: {len(repos)} "
              f"(skipping {skipped} repos with no target-city contributors)")
    else:
        repos = all_repos
        print(f"  Could not pre-filter, processing all {len(repos)} repos")

    # Apply sharding: each shard takes a deterministic slice
    if shard_id is not None and shard_total is not None:
        repos = [r for i, r in enumerate(repos) if i % shard_total == (shard_id - 1)]
        print(f"  Shard {shard_id}/{shard_total}: {len(repos)} repos assigned")

    # Repo metadata for 'create' events
    repo_meta = {}
    for _, r in prominent.iterrows():
        repo_meta[r["repo_full_name"]] = {
            "owner": r.get("owner_login", ""),
            "created_at": r.get("created_at", ""),
        }

    # Shard-specific or default output path
    if shard_id is not None:
        out_path = (DATA_RAW / "github" /
                    f"github_repo_participation_events_s{shard_id}.csv")
    else:
        out_path = DATA_RAW / "github" / "github_repo_participation_events.csv"
    done_repos = set()
    existing_rows = []
    if out_path.exists():
        ex = pd.read_csv(out_path, dtype=str)
        done_repos = set(ex["repo_full_name"].unique())
        existing_rows = ex.values.tolist()
        print(f"  Already processed: {len(done_repos)} repos (resuming)")

    todo = [r for r in repos if r not in done_repos]
    print(f"  Repos to process: {len(todo)}")
    if not todo:
        print("  All repos already processed.")
        return

    rows = list(existing_rows)
    start_time = time.time()
    interrupted = False

    try:
        for i, repo in enumerate(todo, 1):
            # --- Create event (from CSV, no API call) ---
            meta = repo_meta.get(repo, {})
            actor_events = {}  # login → (date, type)
            owner = meta.get("owner", "")
            created = meta.get("created_at", "")
            if owner and created:
                actor_events[owner] = (created, "create")

            # --- Commits (serial, rate-limited) ---
            commit_authors = _fetch_repo_commits(repo)
            for login, date in commit_authors.items():
                if login not in actor_events or date < actor_events[login][0]:
                    actor_events[login] = (date, "commit")

            # --- PRs (serial, rate-limited) ---
            pr_authors = _fetch_repo_prs(repo)
            for login, date in pr_authors.items():
                if login not in actor_events or date < actor_events[login][0]:
                    actor_events[login] = (date, "pr_opened")

            # --- Append rows ---
            for login, (date, etype) in actor_events.items():
                rows.append([repo, login, date, etype])

            # --- Checkpoint (frequent saves to minimize data loss) ---
            if i % CHECKPOINT_EVERY == 0 or i == len(todo):
                _save_events(rows, out_path)

            # --- Progress log ---
            if i % LOG_EVERY == 0 or i == len(todo):
                elapsed = time.time() - start_time
                rate = i / elapsed * 3600 if elapsed > 0 else 0
                eta_h = (len(todo) - i) / rate if rate > 0 else 0
                print(f"  [{i}/{len(todo)}] {repo}: "
                      f"{len(commit_authors)} commits, "
                      f"{len(pr_authors)} PRs  "
                      f"({rate:.0f} repos/hr, ETA {eta_h:.1f}h)")

    except KeyboardInterrupt:
        interrupted = True
        print(f"\n  ⚠️  Interrupted! Saving progress ({i}/{len(todo)} done) …")
        _save_events(rows, out_path)
        print(f"  ✅ Progress saved. Re-run to resume from repo #{i}.")

    if not interrupted:
        total_actors = len(rows)
        print(f"\n✅ Participation events: {total_actors} rows "
              f"across {len(done_repos) + len(todo)} repos")


# ──────────────────────────────── Shard merge ────────────────────────────────

def merge_shard_files():
    """Merge shard CSVs into the final participation_events file."""
    import glob as globmod
    pattern = str(DATA_RAW / "github" / "github_repo_participation_events_s*.csv")
    shard_files = sorted(globmod.glob(pattern))
    if not shard_files:
        print("  No shard files found.")
        return

    out_path = DATA_RAW / "github" / "github_repo_participation_events.csv"
    all_rows = []
    seen = set()
    for f in shard_files:
        df = pd.read_csv(f, dtype=str)
        for _, row in df.iterrows():
            key = (row["repo_full_name"], row["actor_login"])
            if key not in seen:
                seen.add(key)
                all_rows.append(row.tolist())
        print(f"  Loaded {len(df)} rows from {Path(f).name}")

    _save_events(all_rows, out_path)
    print(f"\n✅ Merged {len(all_rows)} rows → "
          f"github_repo_participation_events.csv")


# ──────────────────────────────── Main ───────────────────────────────────────

def main():
    import argparse
    parser = argparse.ArgumentParser(description="Step 4: Fetch contributors")
    parser.add_argument(
        "--shard", type=str, default=None, metavar="N/M",
        help="Run only shard N of M (e.g. --shard 1/2). "
             "Each shard writes its own file. Use --merge after all shards finish.")
    parser.add_argument(
        "--merge", action="store_true",
        help="Merge shard files into final participation_events.csv")
    parser.add_argument(
        "--part-c-only", action="store_true",
        help="Skip Part A/B, only run Part C (participation chain)")
    args = parser.parse_args()

    # Parse shard spec
    shard_id, shard_total = None, None
    if args.shard:
        try:
            n, m = args.shard.split("/")
            shard_id, shard_total = int(n), int(m)
            assert 1 <= shard_id <= shard_total
        except (ValueError, AssertionError):
            print("❌ --shard must be N/M where 1 ≤ N ≤ M, e.g. --shard 1/2")
            sys.exit(1)

    # Merge mode
    if args.merge:
        print("=" * 60)
        print("Merging shard files")
        print("=" * 60)
        merge_shard_files()
        return

    print("=" * 60)
    if shard_id:
        print(f"Step 4: Shard {shard_id}/{shard_total}")
    else:
        print("Step 4: Fetch contributors & their locations")
    print("=" * 60)

    if not GITHUB_TOKEN:
        print("⚠️  GITHUB_TOKEN not set. Will be heavily rate-limited.\n"
              "   export GITHUB_TOKEN='ghp_…' before running.")

    if not args.part_c_only:
        # Part A: get contributor lists for all prominent repos
        all_logins = collect_all_contributors()

        # Part B: fetch locations for new contributors
        fetch_missing_locations(all_logins)
    else:
        print("  Skipping Part A/B (--part-c-only)")

    # Part C: build participation chain (Commits + PRs)
    print("\n" + "-" * 60)
    print("Part C: Building participation chain (Commits API + PRs API)")
    if shard_id:
        print(f"         Shard {shard_id} of {shard_total}")
    print("-" * 60)
    collect_participation_events(shard_id=shard_id, shard_total=shard_total)

    print("\n" + "=" * 60)
    if shard_id:
        print(f"Shard {shard_id}/{shard_total} complete.")
        if shard_total > 1:
            print("After ALL shards finish, run:  python step_09_contributors_participation.py --merge")
    else:
        print("Step 4 complete. Ready for step 5 (build core tables).")
    print("=" * 60)


if __name__ == "__main__":
    main()


---
## Step 10/13 — HF base_model 衍生边

**原 `step5b_build_hf_derivation_edges.py` · 须在 Step 11 前运行**

源文件：`ELSE/scripts/step_10_hf_derivation_edges.py`

执行：`import step_10_hf_derivation_edges as _m; _m.main()` 或在终端 `python ELSE/scripts/step_10_hf_derivation_edges.py`。


In [ ]:
"""
Step 5b – Parse the Hugging Face derivation graph from the local
`tags` field that step1b stored on every HF candidate.

Background
----------
HF model cards encode parent-of-derivation relations as tags such as
    base_model:<owner>/<name>
    base_model:finetune:<owner>/<name>
    base_model:quantized:<owner>/<name>
    base_model:adapter:<owner>/<name>
    base_model:merge:<owner>/<name>

We restrict to:
  * descendants whose project_id is in `prominent_projects_master.csv`
    (any HF type, but only `model` carries base_model in practice);
  * ancestors whose project_id is also in the prominent set
    (so the resulting adoption events live within our analysis frame).

Outputs
-------
data/processed/hf_derivation_edges.csv
    descendant_id, ancestor_id, relation,
    descendant_project_id, ancestor_project_id,
    descendant_author, ancestor_author,
    descendant_created_month, ancestor_created_month, lag_months
"""

from __future__ import annotations

import re
import sys
from pathlib import Path
from collections import defaultdict
from typing import Optional

import pandas as pd

from config import DATA_PROCESSED

MASTER = DATA_PROCESSED / "prominent_projects_master.csv"
OUT = DATA_PROCESSED / "hf_derivation_edges.csv"

REL_PATTERN = re.compile(
    r"^base_model:(?:(finetune|quantized|adapter|merge):)?(.+)$",
    re.IGNORECASE,
)


def _parse_base_model_tags(tag_str: str):
    """Return list of (relation, parent_full_id) parsed from stored tags."""
    if not tag_str or pd.isna(tag_str):
        return []
    out = []
    for t in str(tag_str).split("|"):
        t = t.strip()
        m = REL_PATTERN.match(t)
        if m:
            rel = (m.group(1) or "generic").lower()
            parent = m.group(2).strip()
            if "/" in parent:
                out.append((rel, parent))
    return out


def _to_month(ts) -> Optional[int]:
    if not ts or pd.isna(ts):
        return None
    s = str(ts).strip()
    try:
        return int(s[:4]) * 100 + int(s[5:7])
    except (ValueError, IndexError):
        return None


def _month_diff(m_old: int, m_new: int) -> int:
    y1, mo1 = divmod(m_old, 100)
    y2, mo2 = divmod(m_new, 100)
    return (y2 - y1) * 12 + (mo2 - mo1)


def main():
    print("=" * 64)
    print("Step 5b: Parse HF derivation graph from local tags")
    print("=" * 64)

    if not MASTER.exists():
        print(f"❌ Missing {MASTER}. Run step1c first.")
        return

    df = pd.read_csv(MASTER, dtype=str)
    hf = df[df["platform"] == "HuggingFace"].copy()
    print(f"  HF prominent rows: {len(hf)}")

    # Build a lookup: full_id -> (project_id, author, created_month)
    info = {}
    for _, row in hf.iterrows():
        fid = row["full_id"]
        if not fid:
            continue
        info[fid] = {
            "project_id": row["project_id"],
            "author": fid.split("/")[0] if "/" in fid else "",
            "created_month": _to_month(row.get("created_at")),
        }
    print(f"  Lookup index size: {len(info)}")

    # Iterate over all HF rows, parse tags, keep edges where ancestor is in
    # the prominent set as well.
    rel_counter = defaultdict(int)
    keep_edges = []   # list of dict
    seen_pairs = set()  # de-duplicate (descendant_id, ancestor_id)

    for _, row in hf.iterrows():
        child = row["full_id"]
        if not child or child not in info:
            continue
        rels = _parse_base_model_tags(row["tags"])
        if not rels:
            continue

        # Resolve duplicate parents (generic + typed for the same parent):
        # keep the typed relation when present.
        seen_parents = {}
        for rel, parent in rels:
            if parent not in seen_parents or seen_parents[parent] == "generic":
                seen_parents[parent] = rel

        c_info = info[child]
        for parent, rel in seen_parents.items():
            rel_counter[rel] += 1
            if parent not in info:
                continue   # ancestor not in prominent set → skip
            if (child, parent) in seen_pairs:
                continue
            seen_pairs.add((child, parent))

            p_info = info[parent]
            d_month = c_info["created_month"]
            a_month = p_info["created_month"]
            lag = (_month_diff(a_month, d_month)
                   if (a_month is not None and d_month is not None) else None)

            keep_edges.append({
                "descendant_id": child,
                "ancestor_id": parent,
                "relation": rel,
                "descendant_project_id": c_info["project_id"],
                "ancestor_project_id": p_info["project_id"],
                "descendant_author": c_info["author"],
                "ancestor_author": p_info["author"],
                "descendant_created_month": d_month,
                "ancestor_created_month": a_month,
                "lag_months": lag,
            })

    edges_df = pd.DataFrame(keep_edges)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    edges_df.to_csv(OUT, index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n  Raw base_model edges parsed (any relation): "
          f"{sum(rel_counter.values())}")
    for r, n in sorted(rel_counter.items(), key=lambda x: -x[1]):
        print(f"    {r:10s}: {n}")

    print(f"\n  Edges retained (both ends prominent): {len(edges_df)}")
    if len(edges_df) > 0:
        n_descendants = edges_df["descendant_id"].nunique()
        n_ancestors = edges_df["ancestor_id"].nunique()
        print(f"    distinct descendants: {n_descendants}")
        print(f"    distinct ancestors  : {n_ancestors}")
        print(f"    lag months distribution:")
        print(edges_df["lag_months"].describe().to_string())
        print(f"\n    relation breakdown (kept edges):")
        print(edges_df["relation"].value_counts().to_string())
        print(f"\n    top-10 ancestors by in-degree:")
        top = (edges_df.groupby("ancestor_id").size()
               .sort_values(ascending=False).head(10))
        print(top.to_string())

    print(f"\n✅ Saved → {OUT}")


if __name__ == "__main__":
    main()


---
## Step 11/13 — 三张核心分析表

**原 `step5_build_core_tables.py`**

源文件：`ELSE/scripts/step_11_core_tables.py`

执行：`import step_11_core_tables as _m; _m.main()` 或在终端 `python ELSE/scripts/step_11_core_tables.py`。


In [ ]:
"""
Step 5 – Build three core analysis tables from collected data.

Prerequisites
-------------
- data/raw/github/github_candidates.csv                    (from step 1a)
- data/raw/github/github_repo_contributors.csv             (from step 4)
- data/raw/github/github_repo_participation_events.csv     (from step 4, Part C)
- data/raw/github/github_owner_locations.csv               (from step 3a + step 4)
- data/processed/city_list.csv                             (from step 3d)
- data/processed/prominent_projects_master.csv             (from step 1c)
- data/processed/hf_derivation_edges.csv                   (from step 5b, optional)

Outputs (all in data/output/)
-------
1. city_project_adoption_events.csv
2. city_collaboration_edges.csv
3. city_attributes.csv

Logic overview
--------------
- "adoption" comes from two complementary sources:
    * GitHub contributor / participation events (a contributor from city C
      participates in repo R)
    * Hugging Face derivation relations (a model authored by city C
      declares ancestor model A as its base_model → C "adopted" A)
  In both cases, the first month the city appears in connection to the
  project is the adoption month.
- "origination" = the city of the project owner at creation time
  (GitHub repo owner OR HF project author).
- "collaboration edge" = two cities share contributors on the same project
  (GitHub) OR are linked by a derivation relation (HF descendant city → HF
  ancestor city).
- All timestamps are aggregated to YYYYMM monthly granularity.
"""

import sys
import itertools
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

from config import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT

# We need the cleaning functions from step3b to map new contributors
from step_06_clean_map_locations import (
    _build_city_dict, clean_location, match_city, CITY_DICT
)


def _to_month(ts_str):
    """Convert ISO timestamp string to YYYYMM integer."""
    if not ts_str or pd.isna(ts_str):
        return None
    try:
        ts_str = str(ts_str).strip()
        return int(ts_str[:4]) * 100 + int(ts_str[5:7])
    except (ValueError, IndexError):
        return None


def load_data():
    """Load and prepare all required datasets."""
    print("  Loading data …")

    # 1. GitHub prominent repos
    gh = pd.read_csv(DATA_RAW / "github" / "github_candidates.csv", dtype=str)
    gh_prom = gh[gh["prominent_flag"] == "1"].copy()
    gh_prom["created_month"] = gh_prom["created_at"].apply(_to_month)
    print(f"    Prominent GitHub repos: {len(gh_prom)}")

    # 2. Contributors
    contrib_path = DATA_RAW / "github" / "github_repo_contributors.csv"
    if contrib_path.exists():
        contrib = pd.read_csv(contrib_path, dtype=str)
        print(f"    Contributor records: {len(contrib)}")
    else:
        print("    ⚠️  No contributor file found. Run step4 first.")
        contrib = pd.DataFrame(columns=["repo_full_name", "contributor_login", "contributions"])

    # 3. User locations — GitHub owners + HF authors merged into one table
    loc_path = DATA_RAW / "github" / "github_owner_locations.csv"
    loc = pd.read_csv(loc_path, dtype=str).fillna("")
    print(f"    GitHub user location records: {len(loc)}")

    hf_loc_path = DATA_RAW / "huggingface" / "hf_author_locations.csv"
    if hf_loc_path.exists():
        hf_loc = pd.read_csv(hf_loc_path, dtype=str).fillna("")
        # Conform to the schema of github_owner_locations.csv
        # (login, location, type) so build_user_city_map works for both.
        hf_loc_renamed = pd.DataFrame({
            "login": hf_loc["author"],
            "location": hf_loc["raw_location"],
            "type": hf_loc["entity_type"].replace({
                "organization": "Organization",
                "user": "User",
                "unknown": "User",
            }),
        })
        loc = pd.concat([loc, hf_loc_renamed], ignore_index=True)
        print(f"    HF author location records  : {len(hf_loc_renamed)}")
        print(f"    Combined location records   : {len(loc)}")
    else:
        print("    ⚠️  No hf_author_locations.csv — HF authors will be skipped.")

    # 4. City list
    cities = pd.read_csv(DATA_PROCESSED / "city_list.csv", dtype=str)
    city_set = set(cities["matched_city"].str.strip().tolist())
    print(f"    Target cities: {len(city_set)}")

    # 5. Participation events (from step 4 Part C – Commits/PRs API)
    pe_path = DATA_RAW / "github" / "github_repo_participation_events.csv"
    if pe_path.exists():
        pe_df = pd.read_csv(pe_path, dtype=str)
        first_event_map = {}
        for _, row in pe_df.iterrows():
            m = _to_month(row.get("first_event_at", ""))
            if m:
                key = (row["repo_full_name"], row["actor_login"])
                if key not in first_event_map or m < first_event_map[key]:
                    first_event_map[key] = m
        print(f"    Participation events loaded: {len(first_event_map)}")
    else:
        first_event_map = {}
        print("    ⚠️  No participation events file found. "
              "Will use approximation for adoption lag.")

    return gh_prom, contrib, loc, cities, city_set, first_event_map


def load_hf_data():
    """Load HF prominent projects + derivation edges parsed from local tags.

    Returns
    -------
    hf_prom : DataFrame
        Prominent HF rows from prominent_projects_master.csv with helper
        columns: ``author`` and ``created_month``.
    derivation_df : DataFrame
        Output of step5b (descendant_id → ancestor_id edges with relation,
        author and month info). Empty DataFrame if step5b hasn't been run.
    """
    print("\n  Loading HF data …")

    master_path = DATA_PROCESSED / "prominent_projects_master.csv"
    if not master_path.exists():
        print("    ⚠️  prominent_projects_master.csv missing; skipping HF.")
        return pd.DataFrame(), pd.DataFrame()

    master = pd.read_csv(master_path, dtype=str)
    hf_prom = master[master["platform"] == "HuggingFace"].copy()
    hf_prom["author"] = hf_prom["full_id"].fillna("").str.split("/").str[0]
    hf_prom["created_month"] = hf_prom["created_at"].apply(_to_month)
    print(f"    Prominent HF projects: {len(hf_prom)}")

    deriv_path = DATA_PROCESSED / "hf_derivation_edges.csv"
    if deriv_path.exists():
        deriv = pd.read_csv(deriv_path, dtype=str)
        for col in ("descendant_created_month",
                    "ancestor_created_month",
                    "lag_months"):
            if col in deriv.columns:
                deriv[col] = pd.to_numeric(deriv[col], errors="coerce")
        print(f"    HF derivation edges  : {len(deriv)}")
    else:
        print("    ⚠️  hf_derivation_edges.csv missing — run step5b first.")
        deriv = pd.DataFrame()

    return hf_prom, deriv


def build_user_city_map(loc_df, city_set):
    """Map login → city (only cities in our curated list)."""
    _build_city_dict()
    user_city = {}

    for _, row in loc_df.iterrows():
        login = row["login"]
        raw = row["location"]
        if not raw.strip():
            continue
        cleaned = clean_location(raw)
        if cleaned is None:
            continue
        result = match_city(cleaned)
        if result:
            city_name = result[0]
            if city_name in city_set:
                user_city[login] = city_name

    print(f"    Users mapped to target cities: {len(user_city)}")
    return user_city


# ═══════════════════════════════════════════════════════════════════════════════
# Table 1: city_project_adoption_events
# ═══════════════════════════════════════════════════════════════════════════════

def build_adoption_events(gh_prom, contrib, user_city, first_event_map):
    """
    For each (city, project) pair, determine:
      - whether the city is the originator (owner city at creation)
      - the first month a contributor from that city appeared
      - the adoption lag relative to global origin month

    Uses real participation event dates from Commits/PRs API when available;
    falls back to created_month / created_month+1 approximation otherwise.
    """
    print("\n  Building city_project_adoption_events …")

    repo_owner = dict(zip(gh_prom["repo_full_name"], gh_prom["owner_login"]))
    repo_created = dict(zip(gh_prom["repo_full_name"],
                            gh_prom["created_month"].astype("Int64")))

    city_project = defaultdict(dict)  # repo → {city: first_month}
    api_hits = 0
    api_misses = 0

    # Owner = originator
    for repo in gh_prom["repo_full_name"]:
        owner = repo_owner.get(repo, "")
        cm = repo_created.get(repo)
        if owner in user_city and cm and not pd.isna(cm):
            city = user_city[owner]
            real_month = first_event_map.get((repo, owner))
            city_project[repo][city] = real_month if real_month else int(cm)

    # Source 1: Contributors API records (commit-based contributors)
    for _, row in contrib.iterrows():
        repo = row["repo_full_name"]
        login = row["contributor_login"]
        if login not in user_city:
            continue
        if repo not in repo_created:
            continue
        city = user_city[login]
        cm = repo_created.get(repo)
        if cm is None or pd.isna(cm):
            continue
        cm = int(cm)

        real_month = first_event_map.get((repo, login))
        if real_month:
            contrib_month = real_month
            api_hits += 1
        else:
            contrib_month = cm if (login == repo_owner.get(repo)) else cm + 1
            if contrib_month % 100 > 12:
                contrib_month = (contrib_month // 100 + 1) * 100 + 1
            api_misses += 1

        if city not in city_project[repo] or contrib_month < city_project[repo][city]:
            city_project[repo][city] = contrib_month

    # Source 2: Participation events (includes PR-only contributors
    # not captured by the Contributors API)
    pr_only_added = 0
    for (repo, login), event_month in first_event_map.items():
        if login not in user_city:
            continue
        if repo not in repo_created:
            continue
        city = user_city[login]
        if city not in city_project.get(repo, {}):
            city_project[repo][city] = event_month
            pr_only_added += 1
        elif event_month < city_project[repo][city]:
            city_project[repo][city] = event_month

    print(f"    Participation event hits: {api_hits}, "
          f"fallback approximations: {api_misses}, "
          f"PR-only additions: {pr_only_added}")

    # --- Build event rows ---
    rows = []
    for repo, city_months in city_project.items():
        cm = repo_created.get(repo)
        if cm is None or pd.isna(cm):
            continue
        global_origin = int(cm)
        owner = repo_owner.get(repo, "")
        owner_city = user_city.get(owner, "")

        for city, first_month in city_months.items():
            lag = _month_diff(global_origin, first_month)
            rows.append({
                "city": city,
                "project_id": repo,
                "global_origin_month": global_origin,
                "city_first_adoption_month": first_month,
                "lag": lag,
                "is_originator": int(city == owner_city),
            })

    df = pd.DataFrame(rows)

    neg_count = (df["lag"] < 0).sum()
    if neg_count > 0:
        df = df[df["lag"] >= 0].reset_index(drop=True)
        print(f"    Excluded {neg_count} events with negative lag "
              "(forked repos / code migrations with inherited commit histories)")

    print(f"    Adoption events: {len(df)} rows "
          f"({df['city'].nunique()} cities, {df['project_id'].nunique()} projects)")
    return df


def _month_diff(m1, m2):
    """Number of months between YYYYMM integers."""
    y1, mo1 = divmod(m1, 100)
    y2, mo2 = divmod(m2, 100)
    return (y2 - y1) * 12 + (mo2 - mo1)


# ═══════════════════════════════════════════════════════════════════════════════
# Table 1 (HF supplement): origination + derivation-based adoption
# ═══════════════════════════════════════════════════════════════════════════════

def build_hf_adoption_events(hf_prom, deriv_df, user_city):
    """
    Construct HF adoption events from two sources:

    * Origination — every prominent HF project contributes one row at its
      author's city, with lag = 0 and is_originator = 1.
    * Derivation — for each prominent → prominent base_model edge, the
      descendant author's city is recorded as adopting the ancestor
      project; adoption month = descendant_created_month, lag relative to
      ancestor_created_month.

    Multiple events on the same (city, project_id) are collapsed to the
    earliest month.
    """
    print("\n  Building HF adoption events …")

    if hf_prom.empty:
        return pd.DataFrame(columns=[
            "city", "project_id", "global_origin_month",
            "city_first_adoption_month", "lag", "is_originator",
        ])

    project_origin = {}      # project_id -> origin_month
    project_owner_city = {}  # project_id -> owner_city
    # (city, project_id) -> first_adoption_month
    pair_first_month = {}

    # ── Origination events ───────────────────────────────────────────────────
    n_origin_with_city = 0
    for _, row in hf_prom.iterrows():
        pid = row["project_id"]
        cm = row["created_month"]
        if cm is None or pd.isna(cm):
            continue
        cm = int(cm)
        project_origin[pid] = cm

        author = row["author"]
        city = user_city.get(author)
        if not city:
            continue
        project_owner_city[pid] = city
        key = (city, pid)
        if key not in pair_first_month or cm < pair_first_month[key]:
            pair_first_month[key] = cm
        n_origin_with_city += 1

    print(f"    HF projects with author city: {n_origin_with_city} "
          f"({n_origin_with_city/len(hf_prom)*100:.1f}% of {len(hf_prom)})")

    # ── Derivation-based adoption events ─────────────────────────────────────
    n_deriv_kept = 0
    n_deriv_skipped_city = 0
    n_deriv_skipped_neg = 0
    if not deriv_df.empty:
        for _, edge in deriv_df.iterrows():
            ancestor_pid = edge["ancestor_project_id"]
            if ancestor_pid not in project_origin:
                continue
            d_month = edge.get("descendant_created_month")
            a_month = edge.get("ancestor_created_month")
            if pd.isna(d_month) or pd.isna(a_month):
                continue
            d_month = int(d_month)
            a_month = int(a_month)
            if d_month < a_month:
                # Implausible: descendant predates ancestor. Skip; usually
                # tag noise or cross-fork timestamps.
                n_deriv_skipped_neg += 1
                continue

            descendant_author = edge["descendant_author"]
            descendant_city = user_city.get(descendant_author)
            if not descendant_city:
                n_deriv_skipped_city += 1
                continue

            key = (descendant_city, ancestor_pid)
            if key not in pair_first_month or d_month < pair_first_month[key]:
                pair_first_month[key] = d_month
            n_deriv_kept += 1

    print(f"    HF derivation events kept   : {n_deriv_kept}")
    print(f"      (skipped no descendant city: {n_deriv_skipped_city}, "
          f"skipped negative lag: {n_deriv_skipped_neg})")

    # ── Build event rows ─────────────────────────────────────────────────────
    rows = []
    for (city, pid), first_month in pair_first_month.items():
        origin_m = project_origin.get(pid)
        if origin_m is None:
            continue
        lag = _month_diff(origin_m, first_month)
        rows.append({
            "city": city,
            "project_id": pid,
            "global_origin_month": origin_m,
            "city_first_adoption_month": first_month,
            "lag": lag,
            "is_originator": int(project_owner_city.get(pid) == city),
        })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        n_neg = (df["lag"] < 0).sum()
        if n_neg > 0:
            df = df[df["lag"] >= 0].reset_index(drop=True)
            print(f"    Dropped {n_neg} HF events with negative lag")
    print(f"    HF adoption events total    : {len(df)} rows "
          f"({df['city'].nunique() if len(df) else 0} cities, "
          f"{df['project_id'].nunique() if len(df) else 0} projects)")
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# Table 2: city_collaboration_edges
# ═══════════════════════════════════════════════════════════════════════════════

def build_collaboration_edges(gh_prom, contrib, user_city,
                              hf_prom=None, deriv_df=None):
    """
    Construct undirected city-pair collaboration edges from two sources.

    GitHub: two cities are connected if they share contributors on the
    same prominent repo (counted once per shared repo).

    Hugging Face derivation: two cities are connected if a model authored
    by city A declares a model authored by city B as its base_model
    (counted once per descendant project; intra-city edges discarded).

    Edge weight = total number of shared projects across both sources.
    Monthly snapshot uses repo `created_month` for GitHub and
    `descendant_created_month` for HF.
    """
    print("\n  Building city_collaboration_edges …")

    repo_created = dict(zip(gh_prom["repo_full_name"],
                            gh_prom["created_month"].astype("Int64")))

    # repo_cities[project_key] = (set_of_cities, project_created_month)
    repo_cities = defaultdict(set)
    project_month = {}

    # ── GitHub side ───────────────────────────────────────────────────────────
    repo_owner = dict(zip(gh_prom["repo_full_name"], gh_prom["owner_login"]))
    for repo, owner in repo_owner.items():
        if owner in user_city:
            repo_cities[repo].add(user_city[owner])
        cm = repo_created.get(repo)
        if cm is not None and not pd.isna(cm):
            project_month[repo] = int(cm)

    for _, row in contrib.iterrows():
        repo = row["repo_full_name"]
        login = row["contributor_login"]
        if login in user_city and repo in repo_created:
            repo_cities[repo].add(user_city[login])

    n_gh_projects = len(repo_cities)

    # ── HF derivation side ────────────────────────────────────────────────────
    n_hf_pairs_added = 0
    if hf_prom is not None and deriv_df is not None and not deriv_df.empty:
        hf_origin_month = dict(zip(hf_prom["project_id"],
                                    hf_prom["created_month"]))
        for _, edge in deriv_df.iterrows():
            descendant_pid = edge["descendant_project_id"]
            ancestor_pid = edge["ancestor_project_id"]

            d_city = user_city.get(edge["descendant_author"])
            a_city = user_city.get(edge["ancestor_author"])
            if not d_city or not a_city or d_city == a_city:
                continue

            # Use the descendant project as the "shared project" key so each
            # descendant contributes exactly one shared-project unit
            # between the two cities. Avoid colliding with GitHub repo keys
            # by namespacing.
            key = f"hfderiv::{descendant_pid}"
            repo_cities[key] = {d_city, a_city}
            d_month = edge.get("descendant_created_month")
            if pd.notna(d_month):
                project_month[key] = int(d_month)
            n_hf_pairs_added += 1

    print(f"    Project units contributing edges: "
          f"GitHub repos={n_gh_projects}, HF derivation={n_hf_pairs_added}")

    # ── Aggregate to city pairs ──────────────────────────────────────────────
    edge_counter = defaultdict(lambda: {"weight": 0, "shared_projects": 0,
                                         "months": set()})
    for project_key, cities in repo_cities.items():
        if len(cities) < 2:
            continue
        cm = project_month.get(project_key)
        sorted_cities = sorted(cities)
        for c1, c2 in itertools.combinations(sorted_cities, 2):
            pair = (c1, c2)
            edge_counter[pair]["weight"] += 1
            edge_counter[pair]["shared_projects"] += 1
            if cm is not None:
                edge_counter[pair]["months"].add(int(cm))

    agg_rows = []
    for (c1, c2), info in edge_counter.items():
        agg_rows.append({
            "source_city": c1,
            "target_city": c2,
            "edge_weight": info["weight"],
            "shared_projects": info["shared_projects"],
        })

    df_agg = pd.DataFrame(agg_rows).sort_values("edge_weight",
                                                 ascending=False)

    monthly_rows = []
    for (c1, c2), info in edge_counter.items():
        for m in sorted(info["months"]):
            monthly_rows.append({
                "source_city": c1,
                "target_city": c2,
                "month": m,
                "edge_weight": 1,
            })

    df_monthly = pd.DataFrame(monthly_rows)

    print(f"    Aggregate edges: {len(df_agg)} city pairs")
    print(f"    Monthly edges:   {len(df_monthly)} rows")
    if len(df_agg) > 0:
        print(f"    Top 5 edges:")
        print(df_agg.head(5)[["source_city", "target_city",
                              "edge_weight"]].to_string(index=False))

    return df_agg, df_monthly


# ═══════════════════════════════════════════════════════════════════════════════
# Table 3: city_attributes
# ═══════════════════════════════════════════════════════════════════════════════

def build_city_attributes(adoption_df, edges_agg_df, cities_df, gh_prom, user_city):
    """
    Aggregate city-level indicators:
      - origination_count / rate
      - adoption_count / rate
      - avg_adoption_lag
      - collaboration_count
      - degree, weighted_degree, betweenness (from edge table)
    """
    print("\n  Building city_attributes …")

    import networkx as nx

    city_list = cities_df[["matched_city", "country", "lat", "lon", "entity_count"]].copy()
    city_list = city_list.rename(columns={"matched_city": "city"})
    city_list["lat"] = pd.to_numeric(city_list["lat"], errors="coerce")
    city_list["lon"] = pd.to_numeric(city_list["lon"], errors="coerce")
    city_list["entity_count"] = pd.to_numeric(city_list["entity_count"], errors="coerce")

    # --- Innovation / origination ---
    if len(adoption_df) > 0:
        orig = adoption_df[adoption_df["is_originator"] == 1]
        orig_counts = orig.groupby("city").size().rename("origination_count")

        adopt_counts = adoption_df.groupby("city").agg(
            adoption_count=("project_id", "nunique"),
            avg_lag=("lag", "mean"),
            median_lag=("lag", "median"),
        )
    else:
        orig_counts = pd.Series(dtype=int, name="origination_count")
        adopt_counts = pd.DataFrame(columns=["adoption_count", "avg_lag", "median_lag"])

    city_list = city_list.merge(orig_counts, left_on="city", right_index=True, how="left")
    city_list = city_list.merge(adopt_counts, left_on="city", right_index=True, how="left")

    # --- Collaboration ---
    if len(edges_agg_df) > 0:
        collab_src = edges_agg_df.groupby("source_city")["edge_weight"].sum()
        collab_tgt = edges_agg_df.groupby("target_city")["edge_weight"].sum()
        collab_total = collab_src.add(collab_tgt, fill_value=0).rename("collaboration_count")
    else:
        collab_total = pd.Series(dtype=int, name="collaboration_count")

    city_list = city_list.merge(collab_total, left_on="city", right_index=True, how="left")

    # --- Network centrality ---
    if len(edges_agg_df) > 0:
        G = nx.Graph()
        for _, row in edges_agg_df.iterrows():
            G.add_edge(row["source_city"], row["target_city"],
                       weight=row["edge_weight"])

        degree = pd.Series(dict(G.degree(weight="weight")), name="weighted_degree")
        betweenness = pd.Series(nx.betweenness_centrality(G, weight="weight"),
                                name="betweenness")
        try:
            eigenvector = pd.Series(
                nx.eigenvector_centrality(G, weight="weight", max_iter=500),
                name="eigenvector_centrality")
        except nx.PowerIterationFailedConvergence:
            eigenvector = pd.Series(dtype=float, name="eigenvector_centrality")
    else:
        degree = pd.Series(dtype=float, name="weighted_degree")
        betweenness = pd.Series(dtype=float, name="betweenness")
        eigenvector = pd.Series(dtype=float, name="eigenvector_centrality")

    city_list = city_list.merge(degree, left_on="city", right_index=True, how="left")
    city_list = city_list.merge(betweenness, left_on="city", right_index=True, how="left")
    city_list = city_list.merge(eigenvector, left_on="city", right_index=True, how="left")

    # --- Per-capita rates ---
    pop = city_list["entity_count"].replace(0, np.nan)
    city_list["origination_rate"] = city_list["origination_count"] / pop
    city_list["adoption_rate"] = city_list["adoption_count"] / pop

    # Fill NaN
    fill_zero = ["origination_count", "adoption_count", "collaboration_count",
                 "weighted_degree", "betweenness", "eigenvector_centrality",
                 "origination_rate", "adoption_rate"]
    for col in fill_zero:
        if col in city_list.columns:
            city_list[col] = city_list[col].fillna(0)

    city_list["avg_lag"] = city_list["avg_lag"].round(2)
    city_list["median_lag"] = city_list["median_lag"].round(1)

    # Reorder columns
    col_order = [
        "city", "country", "lat", "lon", "entity_count",
        "origination_count", "origination_rate",
        "adoption_count", "adoption_rate",
        "avg_lag", "median_lag",
        "collaboration_count",
        "weighted_degree", "betweenness", "eigenvector_centrality",
    ]
    city_list = city_list[[c for c in col_order if c in city_list.columns]]

    print(f"    City attributes: {len(city_list)} cities, {len(city_list.columns)} columns")
    return city_list


# ═══════════════════════════════════════════════════════════════════════════════
# Main
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 60)
    print("Step 5: Build three core analysis tables")
    print("=" * 60)

    gh_prom, contrib, loc, cities, city_set, first_event_map = load_data()
    hf_prom, deriv_df = load_hf_data()
    user_city = build_user_city_map(loc, city_set)

    # ── Table 1: GitHub adoption events ───────────────────────────────────────
    gh_adoption = build_adoption_events(gh_prom, contrib, user_city,
                                         first_event_map)
    print(f"    GitHub adoption rows: {len(gh_adoption)}")

    # ── Table 1 (HF supplement) ───────────────────────────────────────────────
    hf_adoption = build_hf_adoption_events(hf_prom, deriv_df, user_city)

    # Combine. (city, project_id) cannot collide because GitHub project_ids
    # use `owner/repo` form and HF project_ids use `hf_*_<full_id>` form.
    if len(hf_adoption) > 0:
        adoption_df = pd.concat([gh_adoption, hf_adoption], ignore_index=True)
    else:
        adoption_df = gh_adoption
    print(f"\n  Combined adoption events: {len(adoption_df)} rows "
          f"(GH={len(gh_adoption)}, HF={len(hf_adoption)})")

    # ── Table 2: collaboration edges (GH + HF) ────────────────────────────────
    edges_agg, edges_monthly = build_collaboration_edges(
        gh_prom, contrib, user_city, hf_prom=hf_prom, deriv_df=deriv_df,
    )

    # ── Table 3: city attributes ──────────────────────────────────────────────
    city_attr = build_city_attributes(adoption_df, edges_agg, cities,
                                       gh_prom, user_city)

    # ── Save ──────────────────────────────────────────────────────────────────
    DATA_OUTPUT.mkdir(parents=True, exist_ok=True)

    adoption_df.to_csv(DATA_OUTPUT / "city_project_adoption_events.csv",
                       index=False)
    print(f"\n✅ Saved → city_project_adoption_events.csv")

    edges_agg.to_csv(DATA_OUTPUT / "city_collaboration_edges.csv",
                     index=False)
    edges_monthly.to_csv(DATA_OUTPUT / "city_collaboration_edges_monthly.csv",
                         index=False)
    print(f"✅ Saved → city_collaboration_edges.csv + monthly variant")

    city_attr.to_csv(DATA_OUTPUT / "city_attributes.csv", index=False)
    print(f"✅ Saved → city_attributes.csv")

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("📊 Final summary")
    print("=" * 60)
    n_gh_proj = adoption_df[~adoption_df["project_id"].str.startswith("hf_",
                            na=False)]["project_id"].nunique()
    n_hf_proj = adoption_df[adoption_df["project_id"].str.startswith("hf_",
                            na=False)]["project_id"].nunique()
    print(f"  city_project_adoption_events : {len(adoption_df)} rows")
    print(f"      distinct GH projects     : {n_gh_proj}")
    print(f"      distinct HF projects     : {n_hf_proj}")
    print(f"  city_collaboration_edges     : {len(edges_agg)} aggregate "
          f"+ {len(edges_monthly)} monthly")
    print(f"  city_attributes              : {len(city_attr)} cities × "
          f"{len(city_attr.columns)} columns")

    if len(city_attr) > 0:
        print(f"\n  Top 10 cities by origination_count:")
        top = city_attr.nlargest(10, "origination_count")
        print(top[["city", "origination_count", "adoption_count",
                    "avg_lag", "weighted_degree"]].to_string(index=False))


if __name__ == "__main__":
    main()


---
## Step 12/13 — 外部社会经济属性

**原 `step6_augment_city_attributes.py`**

源文件：`ELSE/scripts/step_12_augment_external.py`

执行：`import step_12_augment_external as _m; _m.main()` 或在终端 `python ELSE/scripts/step_12_augment_external.py`。


In [ ]:
"""
Step 6 – Augment city_attributes.csv with external city-level data.

Adds the following columns to city_attributes.csv:
  - population          (metro-area population, millions)
  - gdp_per_capita      (country-level, current USD ~2023)
  - education_rate      (country-level tertiary-education gross enrolment ratio)
  - research_capacity   (count of QS-top-500 universities in the city, proxy)
  - digital_infra       (country-level internet-user share, %)
  - rd_expenditure      (country-level R&D spending as % of GDP)
  - timezone_utc        (offset from UTC, hours, derived from longitude)
  - region              (continent / macro-region)

Data sources (approximate values for 2022-2023):
  - City population: UN World Urbanization Prospects / national statistics
  - GDP per capita: World Bank WDI (current USD, 2023)
  - Education: World Bank – gross enrolment ratio, tertiary (%, latest)
  - Internet users: ITU / World Bank (% of population, latest)
  - R&D expenditure: UNESCO / World Bank (% of GDP, latest)
  - Research capacity: QS World University Rankings 2024 (top-500 count per city)

Outputs
-------
  - Overwrites  data/output/city_attributes.csv  with augmented columns
  - Also saves  data/raw/city_attributes/city_external_data.csv  for audit
"""

import sys
from pathlib import Path

import pandas as pd
import numpy as np

from config import DATA_RAW, DATA_OUTPUT

# ═════════════════════════════════════════════════════════════════════════════
# Country-level indicators  (source: World Bank / UNESCO, ~2022-2023)
# Keys are country names matching city_list.csv
# ═════════════════════════════════════════════════════════════════════════════

COUNTRY_DATA = {
    # country: (gdp_per_capita_usd, education_tertiary_%, internet_users_%, rd_pct_gdp)
    "United States":        (76330, 88.2, 92.0, 3.46),
    "China":                (12720, 60.0, 73.0, 2.55),
    "United Kingdom":       (46125, 62.0, 95.0, 2.90),
    "India":                ( 2485, 30.0, 52.0, 0.65),
    "Japan":                (33815, 65.5, 93.0, 3.26),
    "South Korea":          (32255, 98.0, 97.6, 4.93),
    "Germany":              (51380, 72.0, 93.0, 3.13),
    "France":               (42330, 67.0, 92.0, 2.22),
    "Canada":               (52080, 75.0, 93.0, 1.69),
    "Singapore":            (65640, 91.0, 96.0, 1.93),
    "Netherlands":          (57025, 88.0, 95.0, 2.32),
    "Switzerland":          (91930, 63.0, 96.0, 3.37),
    "Australia":            (63530, 113.0, 96.0, 1.68),
    "Sweden":               (55870, 76.0, 97.0, 3.40),
    "Denmark":              (67790, 82.0, 98.0, 2.85),
    "Spain":                (30120, 93.0, 93.0, 1.44),
    "Italy":                (34085, 64.0, 90.0, 1.43),
    "Taiwan":               (32756, 84.0, 90.0, 3.76),
    "Israel":               (52170, 63.0, 90.0, 5.44),
    "Czech Republic":       (27220, 66.0, 88.0, 1.95),
    "Poland":               (18320, 67.0, 87.0, 1.44),
    "Austria":              (52085, 90.0, 93.0, 3.20),
    "Finland":              (50550, 93.0, 96.0, 2.94),
    "Ireland":              (103685, 78.0, 92.0, 1.23),
    "Turkey":               (10674, 99.0, 83.0, 1.09),
    "United Arab Emirates": (49450, 33.0, 99.0, 1.30),
    "Russia":               (11480, 82.0, 88.0, 1.10),
    "Ukraine":              ( 4535, 82.0, 75.0, 0.41),
    "Portugal":             (24570, 72.0, 84.0, 1.69),
    "Vietnam":              ( 4164, 28.5, 79.0, 0.43),
    "Belgium":              (49540, 80.0, 92.0, 3.45),
    "Greece":               (20867, 143.0, 83.0, 1.46),
    "Hungary":              (18390, 47.0, 89.0, 1.62),
    "Bulgaria":             (13980, 79.0, 80.0, 0.85),
    "Indonesia":            ( 4788, 36.0, 66.0, 0.28),
    "Pakistan":             ( 1470, 10.0, 36.5, 0.22),
    "Egypt":                ( 3635, 35.0, 72.0, 0.96),
    "Thailand":             ( 7066, 49.0, 88.0, 1.34),
    "Malaysia":             (12570, 43.0, 97.0, 1.04),
    "Brazil":               ( 8920, 55.0, 84.0, 1.16),
    "Nepal":                ( 1336, 14.0, 52.0, 0.30),
    "Estonia":              (28310, 70.0, 92.0, 1.76),
    "Serbia":               ( 9530, 73.0, 81.0, 0.99),
    "Ethiopia":             (1020,  9.0, 25.0, 0.27),
    "Saudi Arabia":         (32586, 70.0, 99.0, 0.80),
    "Norway":               (87925, 82.0, 98.0, 2.28),
    "Mexico":               (13650, 43.0, 77.0, 0.28),
    "Argentina":            (13710, 108.0, 87.0, 0.46),
    "New Zealand":          (48780, 77.0, 96.0, 1.37),
    "Chile":                (16265, 105.0, 90.0, 0.34),
    "Colombia":             ( 6340, 56.0, 73.0, 0.29),
    "Peru":                 ( 7050, 70.0, 71.0, 0.13),
    "Romania":              (17000, 51.0, 84.0, 0.47),
    "South Africa":         ( 6190, 24.0, 72.0, 0.83),
    "Kenya":                ( 1840, 12.0, 40.0, 0.79),
    "Nigeria":              ( 1560, 10.0, 36.0, 0.13),
    "Philippines":          ( 3670, 36.0, 53.0, 0.16),
}

# ═════════════════════════════════════════════════════════════════════════════
# City metro-area population (millions, ~2023 estimates)
# Sources: UN WUP 2024, national census bureaux, Wikipedia
# ═════════════════════════════════════════════════════════════════════════════

CITY_POPULATION = {
    # --- China ---
    "Beijing": 21.5, "Shanghai": 24.9, "Shenzhen": 17.6, "Hangzhou": 12.2,
    "Guangzhou": 18.7, "Chengdu": 21.0, "Nanjing": 9.4, "Wuhan": 12.3,
    "Xi'an": 13.0, "Hefei": 9.4, "Tianjin": 13.9, "Hong Kong": 7.5,
    "Suzhou": 12.7, "Chongqing": 32.1, "Harbin": 10.0, "Kunming": 8.5,
    "Shenyang": 9.1, "Zhuhai": 2.4, "Jinan": 9.2, "Dalian": 7.5,
    "Xiamen": 5.3,

    # --- United States ---
    "San Francisco": 4.7, "New York": 20.1, "Los Angeles": 13.2,
    "Seattle": 4.0, "Boston": 4.9, "Chicago": 9.6, "Austin": 2.3,
    "Washington": 6.3, "Atlanta": 6.1, "Denver": 2.9, "San Diego": 3.3,
    "San Jose": 2.0, "Philadelphia": 6.2, "Dallas": 7.6, "Houston": 7.1,
    "Miami": 6.2, "Pittsburgh": 2.4, "Palo Alto": 0.07,
    "Mountain View": 0.08, "Sunnyvale": 0.16, "Cupertino": 0.06,
    "Redmond": 0.07, "Portland": 2.5, "Phoenix": 4.9, "Salt Lake City": 1.3,
    "Minneapolis": 3.7, "Ann Arbor": 0.37, "Boulder": 0.33,
    "Berkeley": 0.12, "Santa Clara": 0.13, "Menlo Park": 0.035,
    "Champaign": 0.09, "Princeton": 0.03, "Baltimore": 2.8,
    "Cleveland": 2.1, "Urbana": 0.04, "Cambridge": 0.12,
    "College Park": 0.03, "Orlando": 2.7, "Kansas City": 2.2,
    "Bellevue": 0.15, "Chapel Hill": 0.06, "Saint Louis": 2.8,
    "Tempe": 0.19, "Raleigh": 1.60, "Detroit": 4.39,

    # --- United States (state-level entries → use largest city as proxy) ---
    "California": 39.5, "Oregon": 4.2, "Florida": 22.2, "Texas": 30.0,

    # --- United Kingdom ---
    "London": 9.6, "Cambridge": 0.16, "Oxford": 0.15, "Edinburgh": 0.54,
    "Manchester": 2.8, "Bristol": 0.47, "Sandwell": 0.33, "England": 56.5,

    # --- Germany ---
    "Berlin": 3.6, "Munich": 1.6, "Hamburg": 1.9, "Frankfurt": 2.3,
    "Cologne": 1.1, "Stuttgart": 0.63, "Heidelberg": 0.16,
    "Aachen": 0.25, "Karlsruhe": 0.31, "Nuremberg": 0.52,
    "Hildesheim": 0.10,

    # --- France ---
    "Paris": 12.3, "Lyon": 1.7, "Toulouse": 1.0, "Grenoble": 0.45,
    "Mars": 0.015,

    # --- Netherlands ---
    "Amsterdam": 1.2, "Rotterdam": 1.0, "Delft": 0.10, "Eindhoven": 0.24,
    "Utrecht": 0.36,

    # --- Switzerland ---
    "Zurich": 1.4, "Geneva": 0.64, "Lausanne": 0.42,

    # --- Scandinavia ---
    "Stockholm": 1.6, "Copenhagen": 1.4, "Helsinki": 1.3, "Oslo": 1.1,
    "Gothenburg": 0.60, "Tallinn": 0.45,

    # --- Southern Europe ---
    "Madrid": 6.7, "Barcelona": 5.6, "Lisbon": 2.9,
    "Rome": 4.3, "Milan": 3.2, "Turin": 1.7, "Catania": 0.31,
    "Athens": 3.2,

    # --- Central / Eastern Europe ---
    "Vienna": 2.0, "Prague": 1.3, "Warsaw": 1.8, "Krakow": 0.78,
    "Budapest": 1.8, "Dublin": 1.4, "Brussels": 1.2,
    "Moscow": 12.6, "Saint Petersburg": 5.4,
    "Sofia": 1.3, "Lviv": 0.72, "Belgrade": 1.7,
    "Bucharest": 1.8, "Wroclaw": 0.64,

    # --- Middle East ---
    "Istanbul": 15.6, "Ankara": 5.7,     "Tel Aviv": 4.1, "Haifa": 1.0, "Jerusalem": 0.97,
    "Dubai": 3.5, "Abu Dhabi": 1.5, "Riyadh": 7.7,

    # --- East / Southeast Asia ---
    "Tokyo": 37.4, "Osaka": 19.3, "Kyoto": 1.5, "Nagoya": 9.5, "Tsukuba": 0.25,
    "Seoul": 9.9, "Busan": 3.4, "Daejeon": 1.5,
    "Taipei": 7.0, "Hsinchu": 0.45,
    "Singapore": 5.9,
    "Bangkok": 10.7, "Jakarta": 10.6, "Kuala Lumpur": 8.0,
    "Ho Chi Minh City": 9.3, "Hanoi": 8.4, "Manila": 13.9,

    # --- South Asia ---
    "Bengaluru": 12.3, "Mumbai": 20.7, "New Delhi": 16.8, "Delhi": 16.8,
    "Hyderabad": 10.0, "Chennai": 10.9, "Pune": 7.4, "Kolkata": 15.1,
    "Ahmedabad": 8.3, "Lahore": 13.0, "Karachi": 16.1,
    "Islamabad": 1.2, "Kathmandu": 1.5,

    # --- Oceania ---
    "Sydney": 5.3, "Melbourne": 5.1, "Brisbane": 2.6,
    "Perth": 2.1, "Canberra": 0.47, "Auckland": 1.7,

    # --- Americas ---
    "Toronto": 6.2, "Vancouver": 2.6, "Montreal": 4.3,
    "Ottawa": 1.0, "Calgary": 1.6, "Edmonton": 1.1, "Waterloo": 0.58,
    "Sao Paulo": 22.0, "Rio de Janeiro": 13.5, "Mexico City": 21.8,
    "Buenos Aires": 15.2, "Santiago": 7.1, "Bogota": 10.6,
    "Lima": 10.9,

    # --- Africa ---
    "Cairo": 21.3, "Nairobi": 5.0, "Lagos": 15.9,
    "Cape Town": 4.7, "Johannesburg": 6.1, "Addis Ababa": 5.2,
}

# ═════════════════════════════════════════════════════════════════════════════
# Research capacity: approximate count of QS top-500 universities per city
# Source: QS World University Rankings 2024
# ═════════════════════════════════════════════════════════════════════════════

RESEARCH_CAPACITY = {
    # Major research hubs
    "London": 5, "Boston": 3, "New York": 3, "San Francisco": 2,
    "Los Angeles": 2, "Cambridge": 2, "Tokyo": 3, "Beijing": 3,
    "Shanghai": 2, "Paris": 4, "Seoul": 3, "Singapore": 2,
    "Melbourne": 2, "Sydney": 2, "Toronto": 2, "Zurich": 2,
    "Hong Kong": 3, "Berlin": 2, "Munich": 2, "Edinburgh": 2,

    # Strong research cities (1 top-500 university)
    "Oxford": 1, "Palo Alto": 1, "Seattle": 1, "Chicago": 1,
    "Montreal": 1, "Vancouver": 1, "Austin": 1, "Taipei": 1,
    "Hangzhou": 1, "Nanjing": 1, "Bengaluru": 1, "Delhi": 1,
    "New Delhi": 1, "Stockholm": 1, "Copenhagen": 1, "Helsinki": 1,
    "Amsterdam": 1, "Barcelona": 1, "Madrid": 1, "Lisbon": 1,
    "Dublin": 1, "Brussels": 1, "Vienna": 1, "Prague": 1,
    "Warsaw": 1, "Moscow": 1, "Istanbul": 1, "Tel Aviv": 1,
    "Osaka": 1, "Kyoto": 1, "Brisbane": 1, "Sao Paulo": 1,
    "Buenos Aires": 1, "Mexico City": 1, "Cairo": 1, "Cape Town": 1,
    "Pittsburgh": 1, "San Diego": 1, "Philadelphia": 1,
    "Atlanta": 1, "Washington": 1, "Ann Arbor": 1, "Berkeley": 1,
    "Princeton": 1, "Boulder": 1, "Champaign": 1, "Urbana": 1,
    "College Park": 1, "Chapel Hill": 1, "Geneva": 1, "Lausanne": 1,
    "Delft": 1, "Shenzhen": 1, "Wuhan": 1, "Xi'an": 1,
    "Chengdu": 1, "Guangzhou": 1, "Hefei": 1, "Daejeon": 1,
    "Hsinchu": 1, "Kuala Lumpur": 1, "Bangkok": 1,
    "Milan": 1, "Rome": 1, "Karlsruhe": 1, "Aachen": 1,
    "Saint Petersburg": 1, "Mumbai": 1, "Chennai": 1,
    "Hyderabad": 1, "Kolkata": 1, "Pune": 1, "Waterloo": 1,
    "Busan": 1, "Jakarta": 1, "Hanoi": 1,
}

# ═════════════════════════════════════════════════════════════════════════════
# Region mapping: country → macro-region
# ═════════════════════════════════════════════════════════════════════════════

REGION_MAP = {
    "United States": "North America", "Canada": "North America",
    "Mexico": "Latin America",
    "Brazil": "Latin America", "Argentina": "Latin America",
    "Chile": "Latin America", "Colombia": "Latin America",
    "Peru": "Latin America",

    "United Kingdom": "Europe", "Germany": "Europe", "France": "Europe",
    "Netherlands": "Europe", "Switzerland": "Europe", "Sweden": "Europe",
    "Denmark": "Europe", "Finland": "Europe", "Norway": "Europe",
    "Spain": "Europe", "Italy": "Europe", "Portugal": "Europe",
    "Austria": "Europe", "Czech Republic": "Europe", "Poland": "Europe",
    "Hungary": "Europe", "Ireland": "Europe", "Belgium": "Europe",
    "Greece": "Europe", "Romania": "Europe", "Bulgaria": "Europe",
    "Estonia": "Europe", "Serbia": "Europe",
    "Russia": "Europe", "Ukraine": "Europe",

    "China": "East Asia", "Japan": "East Asia",
    "South Korea": "East Asia", "Taiwan": "East Asia",
    "Hong Kong": "East Asia",

    "Singapore": "Southeast Asia", "Thailand": "Southeast Asia",
    "Vietnam": "Southeast Asia", "Indonesia": "Southeast Asia",
    "Malaysia": "Southeast Asia", "Philippines": "Southeast Asia",

    "India": "South Asia", "Pakistan": "South Asia",
    "Nepal": "South Asia", "Bangladesh": "South Asia",

    "Israel": "Middle East", "Turkey": "Middle East",
    "United Arab Emirates": "Middle East", "Saudi Arabia": "Middle East",
    "Egypt": "Middle East",

    "Australia": "Oceania", "New Zealand": "Oceania",

    "Kenya": "Africa", "Nigeria": "Africa",
    "South Africa": "Africa", "Ethiopia": "Africa",
}


def compute_timezone_utc(lon):
    """Approximate UTC offset from longitude (±0.5h rounding)."""
    if pd.isna(lon):
        return np.nan
    return round(lon / 15.0 * 2) / 2  # round to nearest 0.5


def main():
    print("=" * 60)
    print("Step 6: Augment city_attributes with external data")
    print("=" * 60)

    # ── Load current city_attributes ──
    attr_path = DATA_OUTPUT / "city_attributes.csv"
    df = pd.read_csv(attr_path)
    print(f"  Loaded {len(df)} cities from {attr_path}")

    # ── Handle duplicate city name (Cambridge UK vs US) ──
    # Identify by country
    existing_cities = set(zip(df["city"], df["country"]))

    # ── Add population ──
    def get_population(row):
        city, country = row["city"], row["country"]
        if city in CITY_POPULATION:
            return CITY_POPULATION[city]
        return np.nan

    df["population_million"] = df.apply(get_population, axis=1)

    # ── Add country-level indicators ──
    def get_country_data(country, idx):
        if country in COUNTRY_DATA:
            return COUNTRY_DATA[country][idx]
        return np.nan

    df["gdp_per_capita"] = df["country"].apply(lambda c: get_country_data(c, 0))
    df["education_tertiary_pct"] = df["country"].apply(lambda c: get_country_data(c, 1))
    df["internet_users_pct"] = df["country"].apply(lambda c: get_country_data(c, 2))
    df["rd_expenditure_pct"] = df["country"].apply(lambda c: get_country_data(c, 3))

    # ── Add research capacity ──
    df["research_capacity"] = df["city"].map(RESEARCH_CAPACITY).fillna(0).astype(int)

    # ── Add timezone ──
    df["lon_numeric"] = pd.to_numeric(df["lon"], errors="coerce")
    df["timezone_utc"] = df["lon_numeric"].apply(compute_timezone_utc)
    df = df.drop(columns=["lon_numeric"])

    # ── Add region ──
    df["region"] = df["country"].map(REGION_MAP).fillna("Other")

    # ── Compute per-capita rates using real population ──
    pop = df["population_million"].replace(0, np.nan) * 1_000_000
    df["origination_rate_pop"] = (df["origination_count"] / pop * 1_000_000).round(2)
    df["adoption_rate_pop"] = (df["adoption_count"] / pop * 1_000_000).round(2)
    df["collaboration_rate_pop"] = (df["collaboration_count"] / pop * 1_000_000).round(2)

    # ── Report coverage ──
    pop_missing = df["population_million"].isna().sum()
    gdp_missing = df["gdp_per_capita"].isna().sum()
    region_missing = (df["region"] == "Other").sum()
    print(f"\n  Coverage:")
    print(f"    Population:  {len(df) - pop_missing}/{len(df)} cities")
    print(f"    GDP/capita:  {len(df) - gdp_missing}/{len(df)} cities")
    print(f"    Region:      {len(df) - region_missing}/{len(df)} cities")

    if pop_missing > 0:
        missing = df[df["population_million"].isna()]["city"].tolist()
        print(f"    ⚠️  Missing population: {missing}")

    if gdp_missing > 0:
        missing = df[df["gdp_per_capita"].isna()]["country"].unique().tolist()
        print(f"    ⚠️  Missing GDP country: {missing}")

    # ── Save external data separately for audit ──
    ext_dir = DATA_RAW / "city_attributes"
    ext_dir.mkdir(parents=True, exist_ok=True)
    ext_cols = ["city", "country", "population_million", "gdp_per_capita",
                "education_tertiary_pct", "internet_users_pct", "rd_expenditure_pct",
                "research_capacity", "timezone_utc", "region"]
    df[ext_cols].to_csv(ext_dir / "city_external_data.csv", index=False)
    print(f"\n✅ Saved external data → {ext_dir / 'city_external_data.csv'}")

    # ── Overwrite city_attributes ──
    df.to_csv(attr_path, index=False)
    print(f"✅ Updated city_attributes → {attr_path}")

    # ── Summary ──
    print(f"\n{'=' * 60}")
    print(f"📊 Augmented city_attributes: {len(df)} cities × {len(df.columns)} columns")
    print(f"{'=' * 60}")
    print(f"\n  Columns: {list(df.columns)}")

    print(f"\n  Region distribution:")
    print(df["region"].value_counts().to_string())

    print(f"\n  Top 10 cities by population:")
    top = df.nlargest(10, "population_million")
    print(top[["city", "country", "population_million", "gdp_per_capita",
               "research_capacity"]].to_string(index=False))

    print(f"\n  Descriptive stats for new columns:")
    new_cols = ["population_million", "gdp_per_capita", "education_tertiary_pct",
                "internet_users_pct", "rd_expenditure_pct", "research_capacity"]
    print(df[new_cols].describe().round(2).to_string())


if __name__ == "__main__":
    main()


---
## Step 13/13 — 建模用衍生特征

**原 `step6b_enrich_city_features.py`**

源文件：`ELSE/scripts/step_13_enrich_attributes.py`

执行：`import step_13_enrich_attributes as _m; _m.main()` 或在终端 `python ELSE/scripts/step_13_enrich_attributes.py`。


In [ ]:
"""
Step 6b – Enrich city_attributes.csv with pre-computed analytic features.

Adds columns consumed by downstream analysis (§9.1 K-means, §10.1 regression),
following the same pattern as road_safety_london_final: multi-source aggregations
are computed here (offline script) so the analysis notebook only reads columns and
applies simple log/z-score transforms.

New columns
-----------
  Temporal lag distribution:
    lag_std          — std dev of adoption lag across ALL events per city
    avg_lag_nonorig  — mean lag restricted to non-originator events (§10.1 DV)

  Geographic collaboration reach:
    cross_region_ratio_calc — share of collaboration weight that crosses macro-regions

  Project quality / engagement (from prominent_projects_master.csv):
    pop_top25_share     — mean share of adopted projects in top-25% popularity
    orig_top25_share    — Bayesian-smoothed share of originated projects in top-25%
    lag_quality_corr    — per-city correlation between adoption lag and popularity pctile
    avg_fork_star_ratio — mean fork/star ratio of GitHub projects engaged with

Prerequisites
-------------
  data/output/city_attributes.csv              (from step5 + step6)
  data/output/city_project_adoption_events.csv
  data/output/city_collaboration_edges.csv
  data/processed/prominent_projects_master.csv

Output
------
  Overwrites data/output/city_attributes.csv with 7 new columns appended.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from config import DATA_OUTPUT, DATA_PROCESSED


def main():
    print("=" * 60)
    print("Step 6b: Enrich city_attributes with analytic features")
    print("=" * 60)

    # ── Load inputs ───────────────────────────────────────────────────────
    attr_path = DATA_OUTPUT / "city_attributes.csv"
    df       = pd.read_csv(attr_path)
    adoption = pd.read_csv(DATA_OUTPUT / "city_project_adoption_events.csv")
    edges    = pd.read_csv(DATA_OUTPUT / "city_collaboration_edges.csv")

    print(f"  Loaded {len(df)} cities, {len(adoption)} adoption events, "
          f"{len(edges)} collaboration edges")

    # Drop any columns that might already exist from a prior run
    existing_new = ["lag_std", "avg_lag_nonorig", "cross_region_ratio_calc",
                    "pop_top25_share", "orig_top25_share",
                    "lag_quality_corr", "avg_fork_star_ratio"]
    df = df.drop(columns=[c for c in existing_new if c in df.columns])

    # ── A. Temporal lag features ──────────────────────────────────────────
    # lag_std: std dev of lag across all adoption events per city
    lag_std = adoption.groupby("city")["lag"].std().rename("lag_std")
    df = df.merge(lag_std, left_on="city", right_index=True, how="left")

    # avg_lag_nonorig: mean lag of non-originator events only (used in §10.1)
    nonorig = adoption[adoption["is_originator"] == 0]
    avg_lag_nonorig = (nonorig.groupby("city")["lag"]
                              .mean()
                              .rename("avg_lag_nonorig"))
    df = df.merge(avg_lag_nonorig, left_on="city", right_index=True, how="left")
    print("  ✓ lag_std, avg_lag_nonorig")

    # ── B. Cross-region collaboration ratio ───────────────────────────────
    # Fraction of each city's total edge weight directed to a different macro-region
    city_region = df.set_index("city")["region"].to_dict()
    e = edges.copy()
    e["src_region"] = e["source_city"].map(city_region)
    e["tgt_region"] = e["target_city"].map(city_region)
    e["is_cross"]   = (e["src_region"] != e["tgt_region"]).astype(int)

    cr_src = (e[e["is_cross"] == 1]
              .groupby("source_city")["edge_weight"].sum())
    cr_tgt = (e[e["is_cross"] == 1]
              .groupby("target_city")["edge_weight"].sum())
    t_src  = e.groupby("source_city")["edge_weight"].sum()
    t_tgt  = e.groupby("target_city")["edge_weight"].sum()

    cross_ratio = (
        cr_src.add(cr_tgt, fill_value=0)
        / t_src.add(t_tgt, fill_value=0)
    ).rename("cross_region_ratio_calc")

    df = df.merge(cross_ratio, left_on="city", right_index=True, how="left")
    df["cross_region_ratio_calc"] = df["cross_region_ratio_calc"].fillna(0)
    print("  ✓ cross_region_ratio_calc")

    # ── C. Project quality / engagement features ──────────────────────────
    master_path = DATA_PROCESSED / "prominent_projects_master.csv"
    if not master_path.exists():
        print("  ⚠️  prominent_projects_master.csv not found – skipping quality features")
        for col in ["pop_top25_share", "orig_top25_share",
                    "lag_quality_corr", "avg_fork_star_ratio"]:
            df[col] = np.nan
    else:
        master = pd.read_csv(master_path, dtype=str)
        master["metric_stars"]     = pd.to_numeric(master["metric_stars"],     errors="coerce")
        master["metric_forks"]     = pd.to_numeric(master["metric_forks"],     errors="coerce")
        master["metric_downloads"] = pd.to_numeric(master["metric_downloads"], errors="coerce")

        gh = master[master["platform"] == "GitHub"].copy()
        hf = master[master["platform"] == "HuggingFace"].copy()
        gh["popularity_pctile"] = gh["metric_stars"].rank(pct=True)
        hf["popularity_pctile"] = hf["metric_downloads"].rank(pct=True)
        gh["fork_star_ratio"]   = (gh["metric_forks"]
                                   / gh["metric_stars"].replace(0, np.nan))

        metrics = pd.concat([
            gh[["full_id", "platform", "popularity_pctile", "fork_star_ratio"]],
            hf[["full_id", "platform", "popularity_pctile"]],
        ])

        adopt_m = adoption.merge(
            metrics, left_on="project_id", right_on="full_id", how="left"
        )
        am = adopt_m[adopt_m["popularity_pctile"].notna()].copy()
        am["is_top25"] = (am["popularity_pctile"] >= 0.75).astype(int)

        # C1: pop_top25_share — share of adopted projects that are top-25% by popularity
        pop_top25 = am.groupby("city")["is_top25"].mean().rename("pop_top25_share")
        df = df.merge(pop_top25, left_on="city", right_index=True, how="left")
        df["pop_top25_share"] = df["pop_top25_share"].fillna(
            df["pop_top25_share"].median()
        )

        # C2: orig_top25_share — Bayesian-smoothed share of originated top-25% projects
        orig = am[am["is_originator"] == 1]
        orig_top25_raw    = orig.groupby("city")["is_top25"].mean()
        orig_n            = orig.groupby("city").size()
        global_orig_top25 = float(orig["is_top25"].mean()) if len(orig) > 0 else 0.0
        smooth = pd.DataFrame({"raw": orig_top25_raw, "n": orig_n})
        smooth["orig_top25_share"] = (
            (smooth["raw"] * smooth["n"] + global_orig_top25 * 5)
            / (smooth["n"] + 5)
        )
        df = df.merge(
            smooth[["orig_top25_share"]],
            left_on="city", right_index=True, how="left"
        )
        df["orig_top25_share"] = df["orig_top25_share"].fillna(global_orig_top25)

        # C3: lag_quality_corr — correlation between adoption lag and project popularity
        def _lag_quality_corr(group):
            if len(group) < 5:
                return np.nan
            return group["lag"].corr(group["popularity_pctile"])

        lqc = (am[["city", "lag", "popularity_pctile"]]
                 .groupby("city")
                 .apply(_lag_quality_corr)
                 .rename("lag_quality_corr"))
        df = df.merge(lqc, left_on="city", right_index=True, how="left")
        df["lag_quality_corr"] = df["lag_quality_corr"].fillna(
            df["lag_quality_corr"].median()
        )

        # C4: avg_fork_star_ratio — mean fork/star ratio of GitHub projects
        gh_am = am[am["platform"] == "GitHub"]
        fsr   = (gh_am.groupby("city")["fork_star_ratio"]
                      .mean()
                      .rename("avg_fork_star_ratio"))
        df = df.merge(fsr, left_on="city", right_index=True, how="left")
        df["avg_fork_star_ratio"] = df["avg_fork_star_ratio"].fillna(
            df["avg_fork_star_ratio"].median()
        )
        print("  ✓ pop_top25_share, orig_top25_share, lag_quality_corr, avg_fork_star_ratio")

    # ── Save ──────────────────────────────────────────────────────────────
    df.to_csv(attr_path, index=False)
    print(f"\n✅ Updated city_attributes → {attr_path}")
    print(f"   {len(df)} cities × {len(df.columns)} columns")

    new_cols = ["lag_std", "avg_lag_nonorig", "cross_region_ratio_calc",
                "pop_top25_share", "orig_top25_share",
                "lag_quality_corr", "avg_fork_star_ratio"]
    print(f"\n  New columns summary:")
    print(df[new_cols].describe().round(4).to_string())


if __name__ == "__main__":
    main()
